# Notation and Conventions

This notebook is the **single reference for the mathematical notation used in the course**. It is meant to be read once at the start and consulted whenever a symbol is unclear. It follows the order of the chapters: Chapter 01 (agent architectures), 02 (the RL problem and MDPs), 03 (dynamic programming), 04 (exploration and bandits), 05 (value-based learning), 06 (model-based learning), 07 (neural networks: NFQ, DQN and its extensions), 08 (policy gradient), 09 (actor–critic), 10 (continuous control: DDPG, TD3, SAC), 11 (PPO), 12 (offline RL and Decision Transformers) and 13 (RL for language models). The last two sections collect the symbols whose meaning changes from chapter to chapter and the places where the chapter notes deviate from the conventions stated here.

Seven global rules make almost all the notation predictable.

**1. Timing.** At time step $\displaystyle t$ the agent observes the state $\displaystyle S_t$ and selects the action $\displaystyle A_t$; the environment then returns the reward $\displaystyle R_{t+1}$ and the next state $\displaystyle S_{t+1}$. The reward that *results from acting at* $\displaystyle t$ therefore carries the index $\displaystyle t+1$, **not** $\displaystyle t$. The only exception is the bandit setting of Chapter 04: there is no next state to wait for and, following the standard bandit convention, the reward of the action taken at $\displaystyle t$ is written $\displaystyle R_t$.

**2. Random variable vs. value.** Uppercase $\displaystyle S_t, A_t, R_t$ are *random variables*; lowercase $\displaystyle s, a, r$ are *particular values* they can take. Primed letters $\displaystyle s', a'$ denote the *next* state and the action taken there. A stored transition is written $\displaystyle (s, a, r, s')$, with the **terminal flag** $\displaystyle d$ ($\displaystyle d = 1$ if $\displaystyle s'$ is terminal) when it matters (Chapters 07, 09–12).

**3. True vs. estimated vs. optimal.** Lowercase ($\displaystyle v_\pi, q_\pi, a_\pi, q(a), p, r$) denotes the *true* object defined by the MDP and the policy; uppercase ($\displaystyle V, Q, A, V_k, Q_t(a)$) denotes a *computed or learned estimate* of it; a star ($\displaystyle v_*, q_*, \pi_*, a_*$) denotes the *optimal* object. A hat marks an estimate of a component of the model or of a gradient ($\displaystyle \hat p, \hat r$ in Chapter 06, $\displaystyle \hat A_t$ in Chapters 11 and 13); a tilde marks a *simulated* or *sampled* quantity ($\displaystyle \tilde q_t(a)$ in Chapter 04, $\displaystyle \tilde S, \tilde R$ in Chapter 06, $\displaystyle \tilde a$ in Chapter 10).

**4. Probabilities and expectations.** $\displaystyle P(\cdot)$ is the probability of an event, e.g. $\displaystyle P(A_t = a \mid S_t = s)$; the lowercase $\displaystyle p(\cdot \mid \cdot)$ is reserved for the dynamics function of the MDP (and, with other arguments, for densities such as $\displaystyle p(\tau \mid \pi_\theta)$); $\displaystyle \mathbb{E}[\cdot]$ is the expectation, $\displaystyle \mathbb{E}_\pi[\cdot]$ the expectation when actions are selected by $\displaystyle \pi$, and $\displaystyle \mathbb{E}_{x \sim q}[\cdot]$ the expectation over $\displaystyle x$ drawn from $\displaystyle q$.

**5. Parameters.** From Chapter 07 on, functions are represented by neural networks. $\displaystyle \theta$ always denotes the parameters of the network being optimised *as a policy or online value network* ($\displaystyle Q(s,a;\theta)$, $\displaystyle \pi_\theta$, $\displaystyle \mu_\theta$); $\displaystyle \phi$ denotes the parameters of a *second* network, the critic or value baseline $\displaystyle V(s;\phi)$, $\displaystyle Q_\phi(s,a)$ (and the reward model $\displaystyle r_\phi$ of Chapter 13, where there is no critic); a minus superscript ($\displaystyle \theta^-, \phi^-$) or the subscript "target" denotes a frozen or slowly updated **target copy**. The parameters appear either as a subscript ($\displaystyle \pi_\theta$, $\displaystyle Q_\phi$) or after a semicolon ($\displaystyle V(s;\phi)$, $\displaystyle Q(s,a;\theta)$); the two spellings are equivalent.

**6. Objectives and losses.** $\displaystyle J$ is a **performance measure to be maximised** by gradient ascent ($\displaystyle J(\theta)$, $\displaystyle J(\pi)$); a calligraphic $\displaystyle \mathcal{L}$ is a **loss to be minimised** by gradient descent (the sign is chosen so that descending $\displaystyle \mathcal{L}$ ascends $\displaystyle J$); a plain $\displaystyle L$ is a **surrogate objective** to be maximised ($\displaystyle L_\pi(\pi')$, $\displaystyle L^{\text{CLIP}}$ in Chapter 11). Whenever a target contains the network being trained, it is treated as a constant when differentiating (**semi-gradient**; `detach()` / `torch.no_grad()` in code).

**7. Index letters.** $\displaystyle t$ is an environment time step; $\displaystyle k$ counts the sweeps of an algorithm (Chapter 03) or the updates of a pair (Chapter 05) and never an interaction with the environment; $\displaystyle n$ counts how many times a single action has been selected (Chapter 04) or the number of real rewards before bootstrapping (Chapters 05 and 09); $\displaystyle i, j, l$ are generic summation indices. The section *Overloaded symbols* lists the letters whose meaning depends on the chapter.

## Chapter 01 — Agents and environments

Chapter 01 precedes the RL formalism; its symbols are local to that chapter and written in the language of classical AI.

| Symbol | Meaning |
|---|---|
| $\displaystyle P$ | number of possible percepts at each time step |
| $\displaystyle T$ | maximum length of a percept sequence (the lifetime of the agent, in percepts) |
| $\displaystyle E$ | number of entries of a lookup table implementing the agent function |
| $\displaystyle \mathbf{p}(t), \mathbf{v}(t), \mathbf{a}(t)$ | position, velocity and acceleration of a boid at (continuous) time $\displaystyle t$; bold letters denote vectors |
| $\displaystyle \mathbf{F}(t)$ | total force acting on a boid |
| $\displaystyle \mathbf{F}_{\text{align}}, \mathbf{F}_{\text{cohesion}}, \mathbf{F}_{\text{separation}}, \mathbf{F}_{\text{external}}$ | flocking forces and external force |
| $\displaystyle w_{\text{align}}, w_{\text{cohesion}}, w_{\text{separation}}$ | weights of the flocking forces |
| $\displaystyle m$ | mass of a boid |
| $\displaystyle \Delta t$ | integration time step (Euler method) |

**Size of a table-driven agent:**

$\displaystyle E = \sum_{t=1}^{T} P^t \approx P^T$

**Agent = architecture + program:**

$\displaystyle \text{agent} = \text{architecture} + \text{program}$

**Flocking (Euler integration):**

$\displaystyle \mathbf{a}(t) = \frac{\mathbf{F}(t)}{m}, \qquad \mathbf{F}(t) = w_{\text{align}}\, \mathbf{F}_{\text{align}} + w_{\text{cohesion}}\, \mathbf{F}_{\text{cohesion}} + w_{\text{separation}}\, \mathbf{F}_{\text{separation}} + \mathbf{F}_{\text{external}}$

$\displaystyle \mathbf{v}(t + \Delta t) = \mathbf{v}(t) + \mathbf{a}(t)\, \Delta t, \qquad \mathbf{p}(t + \Delta t) = \mathbf{p}(t) + \mathbf{v}(t)\, \Delta t$

**Terminology bridge.** The vocabulary of Chapter 01 maps onto the RL vocabulary used from Chapter 02 onward:

| Chapter 01 (agents) | Chapters 02–13 (reinforcement learning) |
|---|---|
| percept | observation (the state $\displaystyle S_t$ when the environment is fully observable) |
| performance measure delivered as a numerical signal | reward $\displaystyle R_{t+1}$ |
| agent function / agent program | policy $\displaystyle \pi$ |
| transition model | dynamics function $\displaystyle p(s', r \mid s, a)$ (learned as $\displaystyle \hat p, \hat r$ in Chapter 06) |
| utility | value function $\displaystyle v_\pi(s)$, $\displaystyle q_\pi(s,a)$ |
| known / unknown environment | model-based (planning, Chapters 03 and 06) / model-free (learning, Chapters 04–05 and 07–11) |
| one-shot environment | multi-armed bandit (Chapter 04) |
| sequential environment | Markov decision process (Chapter 02) |
| problem generator of a learning agent | exploration (Chapter 04) |
| critic of a learning agent | the critic network of actor–critic methods (Chapter 09) and the reward model of Chapter 13 |

## Chapter 02 — The agent–environment interface

| Symbol | Meaning |
|---|---|
| $\displaystyle t = 0, 1, 2, \ldots$ | discrete time step |
| $\displaystyle \mathcal{S}$ | **state space**, the set of all states |
| $\displaystyle \mathcal{A}$, $\displaystyle \mathcal{A}(s)$ | **action space**; the set of actions available in state $\displaystyle s$ |
| $\displaystyle \mathcal{R} \subset \mathbb{R}$ | set of possible rewards |
| $\displaystyle S_t \in \mathcal{S}$ | state at time $\displaystyle t$ |
| $\displaystyle A_t \in \mathcal{A}(S_t)$ | action selected at time $\displaystyle t$ |
| $\displaystyle R_{t+1} \in \mathcal{R}$ | reward received as a consequence of $\displaystyle A_t$ |
| $\displaystyle S_{t+1} \in \mathcal{S}$ | next state |
| $\displaystyle s, a, r$ | particular values of state, action and reward |
| $\displaystyle s', a'$ | next state and the action taken in it |
| $\displaystyle T$ | final time step of an episode (episodic tasks); $\displaystyle T = \infty$ for continuing tasks |
| $\displaystyle \lvert \mathcal{S} \rvert$, $\displaystyle \lvert \mathcal{A} \rvert$ | number of states and number of actions (finite MDP) |

**Trajectory** produced by the interaction:

$\displaystyle S_0 \overset{A_0}{\longrightarrow} R_1, S_1 \overset{A_1}{\longrightarrow} R_2, S_2 \overset{A_2}{\longrightarrow} R_3, S_3 \overset{A_3}{\longrightarrow} \cdots$

An **episode** runs from an initial state to a terminal state (episodic task); a **continuing task** has no terminal state. Each tuple (state, action, reward, next state) is an **experience** (a *transition*). Gymnasium distinguishes an episode that **terminates** (a terminal state is reached: no future value) from one that is **truncated** by a time limit (the last state does have a future value); the distinction matters from Chapter 07 on.

**Markov property.** Given the present state and action, the past adds no information about the next state and reward:

$\displaystyle P(S_{t+1}=s', R_{t+1}=r \mid S_t=s, A_t=a, S_{t-1}, A_{t-1}, \ldots, S_0, A_0) = P(S_{t+1}=s', R_{t+1}=r \mid S_t=s, A_t=a)$

or, leaving the particular values implicit:

$\displaystyle P(S_{t+1}, R_{t+1} \mid S_t, A_t, S_{t-1}, A_{t-1}, \ldots) = P(S_{t+1}, R_{t+1} \mid S_t, A_t)$

**Stationarity.** The one-step probability

$\displaystyle P(S_t=s', R_t=r \mid S_{t-1}=s, A_{t-1}=a)$

does not depend on $\displaystyle t$. This is what allows the time index to be dropped in the dynamics function below.

## Chapter 02 — Environment dynamics

The course uses **one canonical object**, the four-argument joint dynamics; transition probabilities and expected rewards are all derived from it by marginalisation or conditioning. Spaces are assumed finite (or countable); for continuous spaces $\displaystyle p$ becomes a density and sums become integrals.

| Symbol | Definition | Reading |
|---|---|---|
| $\displaystyle p(s', r \mid s, a)$ | $\displaystyle P(S_t=s', R_t=r \mid S_{t-1}=s, A_{t-1}=a)$ | **dynamics function** (joint probability of next state and reward) |
| $\displaystyle p(s' \mid s, a)$ | $\displaystyle \sum_{r \in \mathcal{R}} p(s', r \mid s, a)$ | **transition function** (marginal over rewards) |
| $\displaystyle p(r \mid s, a)$ | $\displaystyle \sum_{s' \in \mathcal{S}} p(s', r \mid s, a)$ | reward distribution (marginal over next states) |
| $\displaystyle r(s, a)$ | $\displaystyle \mathbb{E}[R_t \mid S_{t-1}=s, A_{t-1}=a]$ | **expected reward** of a state–action pair |
| $\displaystyle r(s, a, s')$ | $\displaystyle \mathbb{E}[R_t \mid S_{t-1}=s, A_{t-1}=a, S_t=s']$ | expected reward of a specific transition |

**Normalisation:**

$\displaystyle \sum_{s' \in \mathcal{S}} \sum_{r \in \mathcal{R}} p(s', r \mid s, a) = 1 \qquad \forall\, s \in \mathcal{S},\ a \in \mathcal{A}(s)$

$\displaystyle \sum_{s' \in \mathcal{S}} p(s' \mid s, a) = 1, \qquad \sum_{r \in \mathcal{R}} p(r \mid s, a) = 1$

**Expected rewards from the dynamics:**

$\displaystyle r(s,a) = \sum_{r \in \mathcal{R}} r\, p(r \mid s, a) = \sum_{r \in \mathcal{R}} r \sum_{s' \in \mathcal{S}} p(s', r \mid s, a)$

$\displaystyle r(s,a,s') = \sum_{r \in \mathcal{R}} r\, \frac{p(s', r \mid s, a)}{p(s' \mid s, a)}, \qquad r(s,a) = \sum_{s' \in \mathcal{S}} p(s' \mid s, a)\, r(s, a, s')$

**Model of the environment** = the dynamics function (equivalently, transition function + reward function). Methods that assume it is known are **model-based** (planning, Chapter 03); methods that learn it from experience are also model-based (Chapter 06); methods that never use it are **model-free** (Chapters 04–05, 07–11). A **distribution model** returns all next states with their probabilities; a **sample model** returns one next state and reward drawn with the right probabilities (Chapter 06).

**In code** (Chapters 02–03, 06), the dynamics of a Gymnasium environment are stored in the dictionary `env.unwrapped.P`: `P[s][a]` is the list of all possible transitions from the pair $\displaystyle (s, a)$, each as a tuple `(prob, next_state, reward, done)`, i.e. $\displaystyle \big( p(s', r \mid s, a),\ s',\ r,\ \text{terminal?} \big)$.

## Chapter 02 — Return and discounting

| Symbol | Meaning |
|---|---|
| $\displaystyle G_t$ | **return** from time $\displaystyle t$ (Chapter 05 writes it $\displaystyle G_{t:T}$ when the horizon must be explicit) |
| $\displaystyle \gamma \in [0,1]$ | **discount factor** |
| $\displaystyle R_{\max}$ | bound on the absolute value of the rewards |

**Undiscounted return** (sum of the rewards that follow time $\displaystyle t$):

$\displaystyle G_t = R_{t+1} + R_{t+2} + R_{t+3} + \cdots = \sum_{k=0}^{\infty} R_{t+k+1}$

**Discounted return:**

$\displaystyle G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^{k} R_{t+k+1}$

For an episodic task the sum stops at the final step $\displaystyle T$ (equivalently, all rewards after $\displaystyle T$ are zero):

$\displaystyle G_t = \sum_{k=0}^{T-t-1} \gamma^{k} R_{t+k+1}$

**Boundedness** (for $\displaystyle \gamma < 1$ and $\displaystyle \lvert R_t \rvert \le R_{\max}$):

$\displaystyle \lvert G_t \rvert \le R_{\max} \sum_{k=0}^{\infty} \gamma^k = \frac{R_{\max}}{1-\gamma}$

**Recursive form** (the basis of the Bellman equations and of all bootstrapping; with $\displaystyle G_T = 0$ it computes all the returns of an episode in one backward pass):

$\displaystyle G_t = R_{t+1} + \gamma\, G_{t+1}$

Here $\displaystyle k$ is a plain summation index; it should not be confused with the sweep counter of Chapter 03 or the number of arms of Chapter 04.

## Chapters 02–03 — Plans and policies

| Symbol | Meaning |
|---|---|
| $\displaystyle \pi$ | a **policy**: a decision rule specifying what to do in every state |
| $\displaystyle \pi(a \mid s)$ | probability of selecting action $\displaystyle a$ in state $\displaystyle s$ (**stochastic** policy) |
| $\displaystyle \pi(s)$ | the action selected in state $\displaystyle s$ (**deterministic** policy) |
| $\displaystyle \pi'$ | an **improved** policy obtained from $\displaystyle \pi$ (Chapter 03); in Chapter 11, a candidate new policy |
| $\displaystyle \pi_0, \pi_1, \pi_2, \ldots$ | the sequence of policies produced by policy iteration |
| $\displaystyle \pi_*$ | an **optimal** policy |

**Stochastic policy:**

$\displaystyle \pi(a \mid s) = P(A_t = a \mid S_t = s), \qquad \sum_{a \in \mathcal{A}(s)} \pi(a \mid s) = 1$

A deterministic policy is the special case in which $\displaystyle \pi(a \mid s) = 1$ for $\displaystyle a = \pi(s)$ and $\displaystyle 0$ otherwise.

A **plan** is a fixed sequence of actions and is sufficient only in deterministic environments. A **policy** prescribes an action in every state the agent may encounter and is what the agent learns.

**Ordering of policies** (Chapter 03):

$\displaystyle \pi \ge \pi' \iff v_\pi(s) \ge v_{\pi'}(s) \quad \forall\, s \in \mathcal{S}, \qquad \pi_* \ge \pi \quad \forall\, \pi$

**Greedy policy** with respect to a value function $\displaystyle v$: in every state, select an action that maximises the one-step lookahead

$\displaystyle \pi'(s) \in \arg\max_{a} \sum_{s', r} p(s', r \mid s, a)\, \big[ r + \gamma\, v(s') \big]$

**In code**, a policy is a Python function `pi(s)` returning an action; the FrozenLake policies of Chapters 02–03 are named Go-get-it, Careful, Careful+, Careful++ and poor. From Chapter 07 on, `greedy_pi` and `stochastic_pi` denote the greedy and the sampling policy derived from a network.

## Chapter 03 — Value functions and Bellman equations

| Symbol | Name | Definition |
|---|---|---|
| $\displaystyle v_\pi(s)$ | **state-value function** (V-function) of $\displaystyle \pi$ | $\displaystyle \mathbb{E}_\pi[G_t \mid S_t = s]$ |
| $\displaystyle q_\pi(s,a)$ | **action-value function** (Q-function) of $\displaystyle \pi$ | $\displaystyle \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a]$ |
| $\displaystyle a_\pi(s,a)$ | **advantage function** of $\displaystyle \pi$ (introduced in Chapter 07, used from Chapter 08 on) | $\displaystyle q_\pi(s,a) - v_\pi(s)$ |
| $\displaystyle v_*(s)$ | **optimal state-value function** | $\displaystyle \max_\pi v_\pi(s)$ |
| $\displaystyle q_*(s,a)$ | **optimal action-value function** | $\displaystyle \max_\pi q_\pi(s,a)$ |
| $\displaystyle V(s)$, $\displaystyle V_k(s)$ | computed **approximation** of $\displaystyle v_\pi$ or $\displaystyle v_*$ (after $\displaystyle k$ sweeps) | array `V[s]` |
| $\displaystyle Q(s,a)$, $\displaystyle Q_k(s,a)$ | computed **approximation** of $\displaystyle q_\pi$ or $\displaystyle q_*$ | array `Q[s, a]` |

**Bellman expectation equations** (value of a fixed policy):

$\displaystyle v_\pi(s) = \sum_{a} \pi(a \mid s) \sum_{s', r} p(s', r \mid s, a)\, \big[ r + \gamma\, v_\pi(s') \big]$

$\displaystyle q_\pi(s,a) = \sum_{s', r} p(s', r \mid s, a)\, \Big[ r + \gamma \sum_{a'} \pi(a' \mid s')\, q_\pi(s', a') \Big]$

**Relations between $\displaystyle v_\pi$ and $\displaystyle q_\pi$:**

$\displaystyle q_\pi(s,a) = \sum_{s', r} p(s', r \mid s, a)\, \big[ r + \gamma\, v_\pi(s') \big] = \mathbb{E}\big[ R_{t+1} + \gamma\, v_\pi(S_{t+1}) \mid S_t = s, A_t = a \big], \qquad v_\pi(s) = \sum_{a} \pi(a \mid s)\, q_\pi(s,a)$

and, for a deterministic policy, $\displaystyle v_\pi(s) = q_\pi(s, \pi(s))$. The advantage $\displaystyle a_\pi(s,a) = q_\pi(s,a) - v_\pi(s)$ therefore has zero mean under the policy: $\displaystyle \sum_a \pi(a \mid s)\, a_\pi(s,a) = 0$.

**Bellman optimality equations** (value of the best policy; the average over actions is replaced by a maximum):

$\displaystyle v_*(s) = \max_{a} \sum_{s', r} p(s', r \mid s, a)\, \big[ r + \gamma\, v_*(s') \big]$

$\displaystyle q_*(s,a) = \sum_{s', r} p(s', r \mid s, a)\, \big[ r + \gamma \max_{a'} q_*(s', a') \big]$

**Relations between optimal quantities:**

$\displaystyle q_*(s,a) = \mathbb{E}\big[ R_{t+1} + \gamma\, v_*(S_{t+1}) \mid S_t = s, A_t = a \big], \qquad v_*(s) = \max_{a} q_*(s,a), \qquad \pi_*(s) \in \arg\max_{a} q_*(s,a)$

**Policy improvement theorem:**

$\displaystyle q_\pi(s, \pi'(s)) \ge v_\pi(s) \quad \forall\, s \in \mathcal{S} \quad \Longrightarrow \quad v_{\pi'}(s) \ge v_\pi(s) \quad \forall\, s \in \mathcal{S}$

## Chapter 03 — Dynamic programming

| Symbol | Meaning |
|---|---|
| $\displaystyle k$ | **sweep counter** of an iterative algorithm; not an environment time step |
| $\displaystyle V_0$ | initial approximation (typically $\displaystyle V_0(s) = 0$) |
| $\displaystyle T_\pi$ | **Bellman expectation operator** for policy $\displaystyle \pi$ |
| $\displaystyle T_*$ | **Bellman optimality operator** |
| $\displaystyle U$ | a second, generic value function (used in the contraction proofs) |
| $\displaystyle \lVert V - U \rVert_\infty$ | **maximum norm**: $\displaystyle \max_{s \in \mathcal{S}} \lvert V(s) - U(s) \rvert$ |
| $\displaystyle \Delta_k$ | largest change produced by sweep $\displaystyle k$ |
| $\displaystyle \theta$ | **stopping tolerance** (a small positive number; `theta` in code). From Chapter 07 on, $\displaystyle \theta$ denotes network parameters |
| $\displaystyle \text{gap}_k$ | maximum distance of $\displaystyle V_k$ from the optimal value function |
| $\displaystyle K$ | number of improvement steps performed by policy iteration |
| $\displaystyle L$ | number of sweeps performed by one policy evaluation |
| $\displaystyle N_{\text{BF}}$, $\displaystyle N_{\text{PI}}$ | number of policies examined by brute-force search / by policy iteration |
| $\displaystyle C_{\text{PI,cycle}}$, $\displaystyle C_{\text{VI,sweep}}$ | cost of one policy-iteration cycle / of one value-iteration sweep |

**Iterative policy evaluation** (Bellman expectation equation used as an update rule):

$\displaystyle V_{k+1}(s) = \sum_{a} \pi(a \mid s) \sum_{s', r} p(s', r \mid s, a)\, \big[ r + \gamma\, V_k(s') \big], \qquad Q_{k+1}(s,a) = \sum_{s', r} p(s', r \mid s, a)\, \Big[ r + \gamma \sum_{a'} \pi(a' \mid s')\, Q_k(s', a') \Big]$

**Bellman operators** (a sweep maps a value function to a value function):

$\displaystyle (T_\pi V)(s) = \sum_{a} \pi(a \mid s) \sum_{s', r} p(s', r \mid s, a)\, \big[ r + \gamma\, V(s') \big], \qquad (T_* V)(s) = \max_{a} \sum_{s', r} p(s', r \mid s, a)\, \big[ r + \gamma\, V(s') \big]$

$\displaystyle V_{k+1} = T_\pi V_k \longrightarrow v_\pi, \qquad V_{k+1} = T_* V_k \longrightarrow v_*$

**Fixed points and contraction** (for $\displaystyle \gamma < 1$):

$\displaystyle T_\pi v_\pi = v_\pi, \qquad T_* v_* = v_*$

$\displaystyle \lVert T_\pi V - T_\pi U \rVert_\infty \le \gamma\, \lVert V - U \rVert_\infty, \qquad \lVert T_* V - T_* U \rVert_\infty \le \gamma\, \lVert V - U \rVert_\infty$

$\displaystyle \lVert V_k - v_\pi \rVert_\infty \le \gamma^k\, \lVert V_0 - v_\pi \rVert_\infty \longrightarrow 0, \qquad \lVert V_k - v_* \rVert_\infty \le \gamma^k\, \lVert V_0 - v_* \rVert_\infty \longrightarrow 0$

Auxiliary inequality used for $\displaystyle T_*$:

$\displaystyle \left\lvert \max_a x_a - \max_a y_a \right\rvert \le \max_a \lvert x_a - y_a \rvert$

**Stopping rule** (both algorithms):

$\displaystyle \Delta_k = \max_{s \in \mathcal{S}} \lvert V_{k+1}(s) - V_k(s) \rvert < \theta$

**Policy improvement** (greedy with respect to $\displaystyle v_\pi$):

$\displaystyle \pi'(s) = \arg\max_{a} q_\pi(s,a) = \arg\max_{a} \sum_{s', r} p(s', r \mid s, a)\, \big[ r + \gamma\, v_\pi(s') \big]$

**Policy iteration:**

$\displaystyle \pi_0 \overset{\text{evaluate}}{\longrightarrow} v_{\pi_0} \overset{\text{improve}}{\longrightarrow} \pi_1 \overset{\text{evaluate}}{\longrightarrow} v_{\pi_1} \overset{\text{improve}}{\longrightarrow} \cdots \overset{\text{improve}}{\longrightarrow} \pi_* \overset{\text{evaluate}}{\longrightarrow} v_{\pi_*}$

A policy is **stable** when $\displaystyle \pi'(s) = \pi(s)$ for all $\displaystyle s$; a stable policy is optimal.

**Value iteration** (Bellman optimality equation used as an update rule, followed by greedy extraction of the policy):

$\displaystyle V_{k+1}(s) = \max_{a} \sum_{s', r} p(s', r \mid s, a)\, \big[ r + \gamma\, V_k(s') \big], \qquad \pi_*(s) = \arg\max_{a} \sum_{s', r} p(s', r \mid s, a)\, \big[ r + \gamma\, v_*(s') \big]$

**Convergence monitoring and counts:**

$\displaystyle \text{gap}_k = \max_{s} \lvert v_*(s) - V_k(s) \rvert$

$\displaystyle N_{\text{BF}} = \lvert \mathcal{A} \rvert^{\lvert \mathcal{S} \rvert}, \qquad N_{\text{PI}} = K + 1 \ll N_{\text{BF}}$

$\displaystyle C_{\text{PI,cycle}} = O\big( \lvert \mathcal{S} \rvert^2 (L + \lvert \mathcal{A} \rvert) \big), \qquad C_{\text{VI,sweep}} = O\big( \lvert \mathcal{S} \rvert^2\, \lvert \mathcal{A} \rvert \big)$

**Generalised policy iteration (GPI)**: any interleaving of *evaluation* (making $\displaystyle V$ consistent with $\displaystyle \pi$) and *improvement* (making $\displaystyle \pi$ greedy with respect to $\displaystyle V$). It is the template of every control method of the course, including actor–critic (Chapter 09), where the critic evaluates and the actor improves.

**In code**: `gamma` is $\displaystyle \gamma$, `theta` is $\displaystyle \theta$, `V` is the array $\displaystyle V_k$, `Q` the array $\displaystyle Q_k$, `delta` is $\displaystyle \Delta_k$, `sweeps` counts $\displaystyle k$; a terminal transition (`done == True`) contributes no future value.

## Chapter 04 — Multi-armed bandits: action values and their estimation

A multi-armed bandit (MAB) is the special case with a **single state** and a **one-step horizon**: the value of an action is its expected reward and there is nothing to discount. Since there is no next state, the reward of the action taken at $\displaystyle t$ is written $\displaystyle R_t$ (bandit convention, see rule 1).

| Symbol | Meaning |
|---|---|
| $\displaystyle k$ | number of arms (actions), $\displaystyle a \in \{1, \ldots, k\}$ |
| $\displaystyle q(a)$ | **true value** of action $\displaystyle a$: $\displaystyle \mathbb{E}[R_t \mid A_t = a]$ |
| $\displaystyle a_*$ | the optimal action, $\displaystyle \arg\max_a q(a)$ |
| $\displaystyle Q_t(a)$ | **estimate** of $\displaystyle q(a)$ available at time $\displaystyle t$ (array `Q` in code) |
| $\displaystyle N_t(a)$ | number of times $\displaystyle a$ has been selected before time $\displaystyle t$ (array `N` in code) |
| $\displaystyle \mathbb{1}_{[\cdot]}$ | indicator function: 1 if the condition holds, 0 otherwise |
| $\displaystyle n$ | number of times a **single, fixed action** has been selected (local index of the incremental derivations) |
| $\displaystyle R_n$ | reward obtained at the $\displaystyle n$-th selection of that action |
| $\displaystyle Q_n$ | estimate of its value after the first $\displaystyle n-1$ rewards; $\displaystyle Q_1$ is the **initial estimate** |
| $\displaystyle \alpha$, $\displaystyle \alpha_n$ | **step size** (constant / used at the $\displaystyle n$-th update) |
| $\displaystyle p$ | exponent of the step-size schedule $\displaystyle \alpha_n = 1/n^p$ |
| $\displaystyle q$, $\displaystyle e_n$, $\displaystyle \eta_n$, $\displaystyle \sigma^2$ | fixed target, estimation error $\displaystyle Q_n - q$, zero-mean reward noise and its variance (Robbins–Monro argument) |

**Objective**: maximise the expected reward; if the values were known, $\displaystyle a_* = \arg\max_a q(a)$.

**Sample-average estimate:**

$\displaystyle Q_t(a) = \frac{\sum_{i=1}^{t-1} R_i\, \mathbb{1}_{[A_i = a]}}{\sum_{i=1}^{t-1} \mathbb{1}_{[A_i = a]}} = \frac{\sum_{i=1}^{t-1} R_i\, \mathbb{1}_{[A_i = a]}}{N_t(a)}$

**Incremental form** and the general **update pattern** of the whole course:

$\displaystyle Q_{n+1} = Q_n + \frac{1}{n}\, (R_n - Q_n)$

$\displaystyle \text{NewEstimate} \leftarrow \text{OldEstimate} + \text{StepSize} \times (\text{Target} - \text{OldEstimate})$

**Constant step size** (exponential recency-weighted average):

$\displaystyle Q_{n+1} = Q_n + \alpha\, (R_n - Q_n), \qquad \alpha \in (0,1]$

$\displaystyle Q_{n+1} = (1-\alpha)^n Q_1 + \sum_{i=1}^{n} \alpha (1-\alpha)^{n-i} R_i, \qquad (1-\alpha)^n + \sum_{i=1}^{n} \alpha (1-\alpha)^{n-i} = 1$

**Variable step size:**

$\displaystyle Q_{n+1} = Q_n + \alpha_n\, (R_n - Q_n), \qquad Q_{n+1} = \left[ \prod_{i=1}^{n} (1-\alpha_i) \right] Q_1 + \sum_{i=1}^{n} \left[ \alpha_i \prod_{j=i+1}^{n} (1-\alpha_j) \right] R_i$

Special cases: $\displaystyle \alpha_i = 1/i$ gives the sample average, $\displaystyle \alpha_i = \alpha$ the recency-weighted average.

**Robbins–Monro conditions** for convergence to a fixed target:

$\displaystyle \sum_{n=1}^{\infty} \alpha_n = \infty, \qquad \sum_{n=1}^{\infty} \alpha_n^2 < \infty$

satisfied by $\displaystyle \alpha_n = 1/n^p$ with $\displaystyle \tfrac{1}{2} < p \le 1$; a constant $\displaystyle \alpha$ violates the second condition (which is desirable in non-stationary problems).

## Chapter 04 — Exploration strategies

| Symbol | Meaning |
|---|---|
| $\displaystyle \varepsilon \in [0,1]$ | probability of exploring (**$\displaystyle \varepsilon$-greedy**); $\displaystyle \varepsilon_{\min}$ is the floor of a decaying schedule |
| $\displaystyle a_{\text{greedy}}$ | the (assumed unique) greedy action, $\displaystyle \arg\max_a Q_t(a)$ |
| $\displaystyle Q_0$ | optimistic initial value (optimistic initialisation) / prior mean (Thompson sampling) |
| $\displaystyle \tau > 0$ | **temperature** (Softmax) |
| $\displaystyle b$ | dummy action index in sums over all actions |
| $\displaystyle c$ | exploration coefficient (**UCB**) |
| $\displaystyle u_t(a)$ | uncertainty bonus of action $\displaystyle a$ at time $\displaystyle t$ (UCB) |
| $\displaystyle \mathcal{D}_t$ | all the evidence (rewards) collected up to time $\displaystyle t$ (Thompson sampling) |
| $\displaystyle \tilde q_t(a)$ | a value of $\displaystyle q(a)$ **sampled** from its posterior (Thompson sampling) |
| $\displaystyle \sigma_0^2$, $\displaystyle \sigma^2$ | prior variance of $\displaystyle q(a)$ / variance of the rewards around $\displaystyle q(a)$ |
| $\displaystyle \mu_t(a)$, $\displaystyle \sigma_t^2(a)$ | posterior mean and variance of $\displaystyle q(a)$ |
| $\displaystyle \lambda_0$, $\displaystyle \lambda_R$, $\displaystyle \lambda_t(a)$ | prior, reward and posterior **precision** (inverse variance) |
| $\displaystyle \mathcal{N}(\mu, \sigma^2)$ | Gaussian distribution with mean $\displaystyle \mu$ and variance $\displaystyle \sigma^2$ |

**Greedy** (pure exploitation; ties broken at random):

$\displaystyle A_t \in \arg\max_{a} Q_t(a)$

**$\displaystyle \varepsilon$-greedy** (explore uniformly among all $\displaystyle k$ actions with probability $\displaystyle \varepsilon$):

$\displaystyle P(A_t = a_{\text{greedy}}) = 1 - \varepsilon + \frac{\varepsilon}{k}, \qquad P(A_t = a) = \frac{\varepsilon}{k} \quad \text{for } a \ne a_{\text{greedy}}$

**Optimistic initial values**: initialise every $\displaystyle Q(a) = Q_0$ to a high value and act greedily; with the sample average, the first update of an action gives $\displaystyle Q(a) \leftarrow Q_0 + (R - Q_0) = R$.

**Softmax (Boltzmann):**

$\displaystyle P(A_t = a) = \frac{e^{Q_t(a)/\tau}}{\sum_{b=1}^{k} e^{Q_t(b)/\tau}}, \qquad \frac{P(A_t = a)}{P(A_t = b)} = e^{(Q_t(a) - Q_t(b))/\tau}$

$\displaystyle \tau \to \infty \ \Longrightarrow\ P(A_t = a) \to \tfrac{1}{k}, \qquad \tau \to 0^+ \ \Longrightarrow\ \text{greedy selection}$

The same softmax, applied to *learned preferences* instead of value estimates, is the policy parameterisation of Chapter 08.

**Upper Confidence Bound (UCB)** (each action is selected once before applying the rule; $\displaystyle c$ multiplies the square root):

$\displaystyle A_t = \arg\max_{a} \Big[ Q_t(a) + c\, u_t(a) \Big], \qquad u_t(a) = \sqrt{\frac{\ln t}{N_t(a)}}$

**Thompson sampling** (Gaussian prior and Gaussian rewards):

$\displaystyle q(a) \sim \mathcal{N}(Q_0, \sigma_0^2), \qquad R \mid q(a) \sim \mathcal{N}(q(a), \sigma^2), \qquad q(a) \mid \mathcal{D}_t \sim \mathcal{N}\big( \mu_t(a), \sigma_t^2(a) \big)$

$\displaystyle \tilde q_t(a) \sim p\big( q(a) \mid \mathcal{D}_t \big), \qquad A_t = \arg\max_{a} \tilde q_t(a)$

$\displaystyle \lambda_0 = \frac{1}{\sigma_0^2}, \qquad \lambda_R = \frac{1}{\sigma^2}, \qquad \mu_t(a) = \frac{\lambda_0 Q_0 + N_t(a)\, \lambda_R\, Q_t(a)}{\lambda_0 + N_t(a)\, \lambda_R}, \qquad \lambda_t(a) = \lambda_0 + N_t(a)\, \lambda_R, \qquad \sigma_t^2(a) = \frac{1}{\lambda_t(a)}$

In $\displaystyle p\big( q(a) \mid \mathcal{D}_t \big)$ the letter $\displaystyle p$ denotes a posterior density, not the MDP dynamics.

## Chapter 04 — Regret and empirical evaluation

| Symbol | Meaning |
|---|---|
| $\displaystyle T$ | horizon: total number of interactions |
| $\displaystyle \Delta_a$ | **gap** of action $\displaystyle a$: expected cost of selecting it once instead of $\displaystyle a_*$ |
| $\displaystyle L_T$ | **cumulative regret** after $\displaystyle T$ interactions |
| $\displaystyle N_T(a)$ | number of times $\displaystyle a$ has been selected over the horizon |
| $\displaystyle \Theta(T)$, $\displaystyle O(\ln T)$ | linear and logarithmic growth of the regret with the horizon |

$\displaystyle \Delta_a = q(a_*) - q(a) \ge 0, \qquad L_T = \sum_{t=1}^{T} \big[ q(a_*) - q(A_t) \big], \qquad \mathbb{E}[L_T] = \sum_{a} \mathbb{E}[N_T(a)]\, \Delta_a$

Linear regret, $\displaystyle L_T = \Theta(T)$, means a non-vanishing cost per step; a good strategy has **sublinear** regret, $\displaystyle L_T / T \to 0$, and the best classical algorithms achieve $\displaystyle L_T = O(\ln T)$.

**Empirical measures** (each averaged over many independently generated bandit problems, at every time step): the *average reward*, the *optimal-action rate* (fraction of runs selecting $\displaystyle a_*$) and the *cumulative regret*. A **parameter study** plots the average reward over a whole run against the main hyper-parameter of each strategy ($\displaystyle \varepsilon$, $\displaystyle \tau$, $\displaystyle c$, $\displaystyle Q_0$, …) on a logarithmic scale. From Chapter 07 on, learning curves are instead the **evaluation score** (average return of the greedy or of the stochastic policy) against training episodes (Chapters 07–08) or environment steps (Chapters 09–11), averaged over several random seeds with the min–max range shown as a band.

## Chapter 05 — Learning values from experience: prediction

The **prediction problem**: estimate $\displaystyle v_\pi$ from sampled episodes without knowing $\displaystyle p(s', r \mid s, a)$. Chapter 05 writes the return with an explicit horizon, $\displaystyle G_{t:T}$, so that every target shows where it stops using real rewards and starts bootstrapping.

| Symbol | Meaning |
|---|---|
| $\displaystyle G_{t:T}$ | **complete (Monte Carlo) return** from $\displaystyle t$ to the end of the episode; the same object as $\displaystyle G_t$ of Chapter 02 |
| $\displaystyle G_{t:t+1}$ | **one-step (TD) target**: one real reward, then $\displaystyle V(S_{t+1})$ |
| $\displaystyle G_{t:t+n}$ | **$\displaystyle n$-step return**: $\displaystyle n$ real rewards, then $\displaystyle V(S_{t+n})$ |
| $\displaystyle G^{\lambda}_{t:T}$ | **$\displaystyle \lambda$-return**: exponentially weighted average of all the $\displaystyle n$-step returns |
| $\displaystyle G^{(j)}(s)$ | the $\displaystyle j$-th return sample observed after a visit to $\displaystyle s$ |
| $\displaystyle N(s)$ | number of visits to $\displaystyle s$ (number of return samples) |
| $\displaystyle V(s)$ | learned estimate of $\displaystyle v_\pi(s)$ (array `V`) |
| $\displaystyle \alpha$ | **step size** (learning rate), constant or decaying |
| $\displaystyle \delta_t$ | **TD error** |
| $\displaystyle \lambda \in [0,1]$ | **trace-decay** parameter: weight of long returns in $\displaystyle G^\lambda$, decay rate of the traces |
| $\displaystyle E(s)$, $\displaystyle E$ | **eligibility trace** of state $\displaystyle s$ / the vector of all traces (array `E`) |
| $\displaystyle \operatorname{RMSE}$ | root mean-squared error of $\displaystyle V$ with respect to $\displaystyle v_\pi$ |

**Returns** (one backward pass computes all of them):

$\displaystyle G_{t:T} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{T-t-1} R_T = R_{t+1} + \gamma\, G_{t+1:T}, \qquad G_{T:T} = 0$

**Monte Carlo prediction** (sample mean of the returns, then its incremental form, then a constant step size):

$\displaystyle V(s) = \frac{1}{N(s)} \sum_{j=1}^{N(s)} G^{(j)}(s), \qquad V(S_t) \leftarrow V(S_t) + \frac{1}{N(S_t)} \big[ G_{t:T} - V(S_t) \big], \qquad V(S_t) \leftarrow V(S_t) + \alpha \big[ G_{t:T} - V(S_t) \big]$

*First-visit* MC uses only the return following the first occurrence of a state in an episode; *every-visit* MC uses all of them.

**TD(0)** (bootstrapping: the unknown $\displaystyle G_{t+1:T}$ is replaced by its current estimate $\displaystyle V(S_{t+1})$):

$\displaystyle G_{t:t+1} = R_{t+1} + \gamma\, V(S_{t+1}), \qquad \delta_t = R_{t+1} + \gamma\, V(S_{t+1}) - V(S_t), \qquad V(S_t) \leftarrow V(S_t) + \alpha\, \delta_t$

**$\displaystyle n$-step TD:**

$\displaystyle G_{t:t+n} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^{n}\, V(S_{t+n}), \qquad V(S_t) \leftarrow V(S_t) + \alpha \big[ G_{t:t+n} - V(S_t) \big]$

$\displaystyle n = 1 \Rightarrow \text{TD(0)}, \qquad t + n \ge T \Rightarrow G_{t:t+n} = G_{t:T} \Rightarrow \text{Monte Carlo}$

**TD($\displaystyle \lambda$), forward view** (weights $\displaystyle 1-\lambda,\ (1-\lambda)\lambda,\ (1-\lambda)\lambda^2, \ldots$ on the 1-, 2-, 3-step returns; the Monte Carlo return takes the remaining weight):

$\displaystyle G^{\lambda}_{t:T} = (1-\lambda) \sum_{n=1}^{T-t-1} \lambda^{n-1} G_{t:t+n} + \lambda^{T-t-1} G_{t:T}, \qquad V(S_t) \leftarrow V(S_t) + \alpha \big[ G^{\lambda}_{t:T} - V(S_t) \big]$

$\displaystyle \lambda = 0 \Rightarrow \text{TD(0)}, \qquad \lambda = 1 \Rightarrow \text{Monte Carlo}$

**TD($\displaystyle \lambda$), backward view** (eligibility traces; $\displaystyle V$ and $\displaystyle E$ are vectors over the states):

$\displaystyle E(s) = 0 \ \forall s \ \text{(start of episode)}, \qquad E(S_t) \leftarrow E(S_t) + 1, \qquad V \leftarrow V + \alpha\, \delta_t\, E, \qquad E \leftarrow \gamma\lambda\, E$

**Bias–variance trade-off.** The MC target is unbiased but has high variance; the TD target has low variance but is biased by the error of $\displaystyle V(S_{t+1})$; $\displaystyle n$ and $\displaystyle \lambda$ interpolate between the two. The same trade-off reappears for advantage estimates in Chapter 09.

| Target for $\displaystyle V(S_t)$ | Expression | Updates after |
|---|---|---|
| Monte Carlo | $\displaystyle G_{t:T}$ | the episode |
| TD(0) | $\displaystyle R_{t+1} + \gamma\, V(S_{t+1})$ | one step |
| $\displaystyle n$-step TD | $\displaystyle G_{t:t+n}$ | $\displaystyle n$ steps |
| TD($\displaystyle \lambda$) | $\displaystyle G^{\lambda}_{t:T}$ (forward) or $\displaystyle \alpha\, \delta_t\, E$ (backward) | the episode / one step |

## Chapter 05 — Learning values from experience: control

The **control problem**: learn $\displaystyle Q$ while improving the policy that collects the experience (generalised policy iteration without a model).

| Symbol | Meaning |
|---|---|
| $\displaystyle Q(s,a)$ | learned estimate of $\displaystyle q_\pi$ or $\displaystyle q_*$ (array `Q[s, a]`) |
| $\displaystyle \pi_k$, $\displaystyle \pi_{k+1}$ | current and improved policy of generalised policy iteration |
| $\displaystyle \pi_{\text{behaviour}}$ | **behaviour policy**: the policy that generates the experience ($\displaystyle \varepsilon$-greedy); written $\displaystyle \pi_\beta$ in Chapter 12 |
| $\displaystyle \pi_{\text{target}}$ | **target policy**: the policy whose value is being learned (greedy in Q-learning) |
| $\displaystyle \varepsilon_t$, $\displaystyle \varepsilon_{\min}$ | exploration rate at time $\displaystyle t$ and the floor of its schedule |
| $\displaystyle \alpha_t$, $\displaystyle \alpha_k(s,a)$, $\displaystyle \alpha_{\min}$ | step size at time $\displaystyle t$ / at the $\displaystyle k$-th update of the pair $\displaystyle (s,a)$ / the floor of its schedule |
| $\displaystyle N_t(s,a)$, $\displaystyle N_k(s,a)$ | number of visits to the pair $\displaystyle (s,a)$ |
| $\displaystyle Q_1$, $\displaystyle Q_2$ | the **two action-value tables** of Double Q-learning (not the initial estimate $\displaystyle Q_1$ of Chapter 04) |
| $\displaystyle A^*$ | the action selected by one table and evaluated by the other (Double Q-learning) |
| $\displaystyle E(s,a)$, $\displaystyle E$ | eligibility trace of a state–action pair / the matrix of all traces |
| $\displaystyle \operatorname{MAE}$ | mean absolute error of $\displaystyle V$ with respect to $\displaystyle v_*$ over the non-terminal states |

**Policy improvement** (greedy with respect to the action values of the current policy; GPI alternates it with evaluation):

$\displaystyle \pi_{k+1}(s) = \arg\max_{a} q_{\pi_k}(s,a), \qquad q_{\pi_k}\big( s, \pi_{k+1}(s) \big) = \max_a q_{\pi_k}(s,a) \ge q_{\pi_k}\big( s, \pi_k(s) \big) = v_{\pi_k}(s)$

$\displaystyle \pi_0 \overset{\text{evaluate}}{\longrightarrow} q_{\pi_0} \overset{\text{improve}}{\longrightarrow} \pi_1 \overset{\text{evaluate}}{\longrightarrow} q_{\pi_1} \overset{\text{improve}}{\longrightarrow} \cdots \overset{\text{improve}}{\longrightarrow} \pi_*$

**On-policy control** (Monte Carlo control and SARSA learn the value of the $\displaystyle \varepsilon$-greedy policy they follow):

$\displaystyle Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha \big[ G_{t:T} - Q(S_t,A_t) \big] \qquad \text{(Monte Carlo control)}$

$\displaystyle G^{\text{SARSA}}_{t:t+1} = R_{t+1} + \gamma\, Q(S_{t+1}, A_{t+1}), \qquad Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha_t \big[ R_{t+1} + \gamma\, Q(S_{t+1}, A_{t+1}) - Q(S_t,A_t) \big] \qquad \text{(SARSA)}$

**Off-policy control** (Q-learning behaves with $\displaystyle \pi_{\text{behaviour}}$ and learns the value of the greedy $\displaystyle \pi_{\text{target}}$):

$\displaystyle A_t \sim \pi_{\text{behaviour}}(S_t), \qquad \pi_{\text{target}}(s) = \arg\max_{a} Q(s,a), \qquad G^{Q}_{t:t+1} = R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a)$

$\displaystyle Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha_t \big[ R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a) - Q(S_t,A_t) \big] \qquad \text{(Q-learning)}$

**Convergence conditions.** *GLIE* (greedy in the limit with infinite exploration), required by on-policy methods; off-policy methods only need the first condition. *Robbins–Monro* for the step size of every pair, required by both:

$\displaystyle N_t(s,a) \to \infty \ \ \forall (s,a), \qquad \pi_t(s) \to \arg\max_{a} Q_t(s,a), \qquad \varepsilon_t \to 0$

$\displaystyle \sum_{k=1}^{\infty} \alpha_k(s,a) = \infty, \qquad \sum_{k=1}^{\infty} \alpha_k^2(s,a) < \infty, \qquad \text{e.g.}\ \alpha_k(s,a) = \frac{1}{N_k(s,a)}$

**Maximisation bias and Double Q-learning** (one table selects, the other evaluates; roles swapped at random):

$\displaystyle \mathbb{E}\Big[ \max_{a} Q(s,a) \Big] \ge \max_{a} \mathbb{E}\big[ Q(s,a) \big]$

$\displaystyle G^{DQ_1}_{t:t+1} = R_{t+1} + \gamma\, Q_2\Big( S_{t+1}, \arg\max_{a} Q_1(S_{t+1}, a) \Big), \qquad G^{DQ_2}_{t:t+1} = R_{t+1} + \gamma\, Q_1\Big( S_{t+1}, \arg\max_{a} Q_2(S_{t+1}, a) \Big)$

**SARSA($\displaystyle \lambda$)** (eligibility traces over pairs; accumulating, replacing, or replacing-with-clearing):

$\displaystyle \delta_t = R_{t+1} + \gamma\, Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t), \qquad Q \leftarrow Q + \alpha\, \delta_t\, E, \qquad E \leftarrow \gamma\lambda\, E$

$\displaystyle E(S_t,A_t) \leftarrow E(S_t,A_t) + 1 \quad \text{(accumulating)}, \qquad E(S_t,A_t) \leftarrow 1 \quad \text{(replacing)}, \qquad E(S_t,a) \leftarrow 0 \ \ \forall a \ne A_t \quad \text{(clearing)}$

**Watkins's Q($\displaystyle \lambda$)** (greedy TD error; traces are cut when the behaviour policy takes a non-greedy action):

$\displaystyle \delta_t = R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a) - Q(S_t, A_t), \qquad Q \leftarrow Q + \alpha\, \delta_t\, E, \qquad E \leftarrow \begin{cases} \gamma\lambda\, E & \text{if } A_{t+1} \text{ is greedy} \\ 0 & \text{otherwise} \end{cases}$

| Method | Target for $\displaystyle Q(S_t,A_t)$ | Policy evaluated |
|---|---|---|
| Monte Carlo control | $\displaystyle G_{t:T}$ | behaviour ($\displaystyle \varepsilon$-greedy) |
| SARSA | $\displaystyle R_{t+1} + \gamma\, Q(S_{t+1}, A_{t+1})$ | behaviour ($\displaystyle \varepsilon$-greedy) |
| Q-learning | $\displaystyle R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a)$ | greedy target |
| Double Q-learning | $\displaystyle R_{t+1} + \gamma\, Q_2\big( S_{t+1}, \arg\max_a Q_1(S_{t+1},a) \big)$ (and symmetric) | greedy target |
| SARSA($\displaystyle \lambda$), Q($\displaystyle \lambda$) | the one-step error $\displaystyle \delta_t$, spread over all eligible pairs by $\displaystyle E$ | as SARSA / as Q-learning |

**In code**: `Q` is the table, `E` the trace matrix, `alpha` and `epsilon` follow exponentially decaying schedules indexed by episode with floors `min_alpha`, `min_epsilon` (so the asymptotic conditions above are deliberately not met in the finite experiments).

## Chapter 06 — Model-based methods

A **model** is an estimate of the dynamics, learned from real experience and then used for **planning**: computations that improve $\displaystyle Q$ without new interaction. Real and simulated experience feed the same Q-learning update (the Dyna architecture).

| Symbol | Meaning |
|---|---|
| $\displaystyle N(s,a,s')$ | number of observed transitions from $\displaystyle (s,a)$ to $\displaystyle s'$ (array `transitions[s][a][s_next]`) |
| $\displaystyle N(s,a)$ | number of times the pair $\displaystyle (s,a)$ has been tried, $\displaystyle \sum_{s'} N(s,a,s')$ |
| $\displaystyle \hat p(s' \mid s, a)$ | **learned transition function** (maximum-likelihood estimate of $\displaystyle p(s' \mid s,a)$) |
| $\displaystyle \hat r(s,a,s')$, $\displaystyle \hat r(s,a)$ | **learned reward function** (estimates of $\displaystyle r(s,a,s')$ and $\displaystyle r(s,a)$; array `rewards[s][a][s_next]`) |
| $\displaystyle \theta_{s'}$, $\displaystyle \lambda$, $\displaystyle \log \mathcal{L}(\theta)$ | unknown transition probabilities, Lagrange multiplier and log-likelihood in the maximum-likelihood derivation (local to that derivation: here $\displaystyle \mathcal{L}$ is a likelihood, not a loss) |
| $\displaystyle n$ | number of **planning** updates per real step (`n_planning`); $\displaystyle n = 0$ is plain Q-learning |
| $\displaystyle \tilde S, \tilde A, \tilde R, \tilde S'$ | **simulated** state, action, reward and next state, drawn from the model (`s_p, a_p, r_p, ns_p` in code) |
| $\displaystyle \kappa$, $\displaystyle \tau(s,a)$ | **Dyna-Q+**: exploration-bonus coefficient and number of time steps since the pair was last tried in the real environment (`kappa`, `t - last_visit[s][a]`) |
| $\displaystyle \beta$ | constant step size of the recency-weighted reward model (`beta`), for non-stationary environments |
| $\displaystyle \delta(s,a)$ | **Bellman residual** of a pair on the learned model (expected backup minus current estimate); the expected version of the TD error $\displaystyle \delta_t$ |
| $\displaystyle P(s,a)$ | **priority** of a pair in prioritized sweeping, $\displaystyle \lvert \delta(s,a) \rvert$ (elsewhere $\displaystyle P$ is the probability of an event) |
| $\displaystyle \theta$ | priority threshold of prioritized sweeping (`theta`); same role as the stopping tolerance of Chapter 03 |
| $\displaystyle \bar s, \bar a$ | a **predecessor** pair of $\displaystyle s$ in the model, i.e. one with $\displaystyle N(\bar s, \bar a, s) > 0$ |

**Model learning** (counts and incremental mean of the reward):

$\displaystyle \hat p(s' \mid s, a) = \frac{N(s,a,s')}{N(s,a)}, \qquad \hat r(s,a,s') \leftarrow \hat r(s,a,s') + \frac{1}{N(s,a,s')} \big[ R_{t+1} - \hat r(s,a,s') \big], \qquad \hat r(s,a) = \sum_{s'} \hat p(s' \mid s, a)\, \hat r(s,a,s')$

**Planning backups** on the learned model (expected / sample; they agree in expectation over $\displaystyle s' \sim \hat p(\cdot \mid s,a)$):

$\displaystyle Q(s,a) \leftarrow \sum_{s'} \hat p(s' \mid s, a) \Big[ \hat r(s,a,s') + \gamma \max_{a'} Q(s',a') \Big], \qquad Q(s,a) \leftarrow Q(s,a) + \alpha \Big[ \hat r(s,a,s') + \gamma \max_{a'} Q(s',a') - Q(s,a) \Big]$

**Value iteration on the learned model** (planning in its purest form):

$\displaystyle V_{k+1}(s) = \max_a \sum_{s'} \hat p(s' \mid s, a) \Big[ \hat r(s,a,s') + \gamma\, V_k(s') \Big]$

**Prioritized sweeping** and **Dyna-Q+**:

$\displaystyle \delta(s,a) = \sum_{s'} \hat p(s' \mid s,a) \Big[ \hat r(s,a,s') + \gamma \max_{a'} Q(s',a') \Big] - Q(s,a), \qquad Q(s,a) \leftarrow Q(s,a) + \alpha\, \delta(s,a), \qquad \tilde R = \hat r(s,a,s') + \kappa \sqrt{\tau(s,a)}, \qquad \hat r(s,a,s') \leftarrow \hat r(s,a,s') + \beta \big[ R_{t+1} - \hat r(s,a,s') \big]$

Terminology (Sutton & Barto, Chapter 8): **direct RL** (update from real experience), **model learning**, **planning** (update from simulated experience), **search control** (choice of the pairs to simulate: uniform over visited pairs in Dyna-Q, along the current policy in trajectory sampling, by decreasing Bellman residual in prioritized sweeping). The Chapter 06 notes use the letters (a)–(e) of the Dyna-Q pseudocode in the code comments.

## Chapter 07 — Generalising with neural networks

Value functions become **parameterised approximators**: learning adjusts parameters instead of filling a table. The chapter covers linear approximation, NFQ, DQN and its extensions (DDQN, prioritized replay, dueling networks).

| Symbol | Meaning |
|---|---|
| $\displaystyle \phi(s,a) \in \mathbb{R}^n$ | **feature vector** of a state–action pair (linear approximation). From Chapter 08 on features are written $\displaystyle x(s,a)$ and $\displaystyle \phi$ denotes critic parameters |
| $\displaystyle n$ | number of features (= parameters) of a linear approximator; a table is the case $\displaystyle n = \lvert \mathcal{S} \rvert\, \lvert \mathcal{A} \rvert$ with one-hot features |
| $\displaystyle \theta$ | **parameters** (weights) of the value network; $\displaystyle \theta_i$ its components |
| $\displaystyle Q_\theta(s,a)$, $\displaystyle Q(s,a;\theta)$ | approximate action-value function (both spellings are used; the network takes the state and outputs one value per action) |
| $\displaystyle \nabla_\theta Q_\theta(s,a)$ | gradient of the prediction with respect to the parameters (the feature vector for a linear approximator, backpropagation for a network) |
| $\displaystyle \theta_{\text{online}}$, $\displaystyle \theta_{\text{target}}$ | parameters of the **online network** (`online_q`, trained at every step) and of the **target network** (`target_q`, a delayed copy used only in the targets); Chapter 10 writes the target copy $\displaystyle \theta^-$ |
| $\displaystyle \mathcal{L}(\theta)$ | **loss function** minimised by gradient descent; $\displaystyle \nabla_\theta \mathcal{L}$ its (semi-)gradient |
| $\displaystyle (s,a,r,s')$, $\displaystyle d$ | a stored **transition**, with its terminal flag (`failure`: 1 if the episode terminated at $\displaystyle s'$, 0 otherwise, also when it was only truncated) |
| $\displaystyle \mathcal{D}$ | **replay memory** (`replay_memory`, a circular buffer) |
| $\displaystyle U(\mathcal{D})$, $\displaystyle P(\mathcal{D})$ | uniform / prioritised sampling of a transition from the memory |
| $\displaystyle N$, $\displaystyle m$ | number of transitions in the memory (`memory_size`) / size of a mini-batch (`batch_size`) |
| $\displaystyle C$ | period, in steps, of the **hard update** of the target network (`target_update_steps`) |
| $\displaystyle \delta$, $\displaystyle \delta_i$ | **TD error** of a transition / of the $\displaystyle i$-th transition in the memory |
| $\displaystyle p_i$, $\displaystyle P(i)$, $\displaystyle w_i$ | **priority**, **sampling probability** and **importance-sampling weight** of the $\displaystyle i$-th transition (PER) |
| $\displaystyle \epsilon$, $\displaystyle \alpha$, $\displaystyle \beta$ | PER constants: priority offset (`priority_epsilon`), prioritisation exponent, importance-sampling exponent (`alpha`, `beta`). Here $\displaystyle \alpha$ is not a step size and $\displaystyle \epsilon$ is not the $\displaystyle \varepsilon$ of $\displaystyle \varepsilon$-greedy |
| $\displaystyle V(s;\theta,\theta_V)$, $\displaystyle A(s,a;\theta,\theta_A)$ | **value head** and **advantage head** of a dueling network (shared trunk $\displaystyle \theta$, head parameters $\displaystyle \theta_V$, $\displaystyle \theta_A$) |
| $\displaystyle a_\pi(s,a)$ | true **advantage function**, $\displaystyle q_\pi(s,a) - v_\pi(s)$ (lowercase by rule 3; $\displaystyle A$ is its estimate) |

**Linear function approximation** (the gradient is the feature vector; the update is the tabular pattern distributed over the active features):

$\displaystyle Q_\theta(s,a) = \theta^\top \phi(s,a) = \sum_{i=1}^{n} \theta_i\, \phi_i(s,a), \qquad \nabla_\theta Q_\theta(s,a) = \phi(s,a), \qquad \theta \leftarrow \theta + \alpha \big[ \text{Target} - Q_\theta(s,a) \big] \phi(s,a)$

**Ideal loss and NFQ loss** (the unknown $\displaystyle q_*$ is replaced by the bootstrapped Q-learning target computed by the same network, fitted for several epochs on a batch of transitions):

$\displaystyle \mathcal{L}(\theta) = \mathbb{E}_{s,a}\Big[ \big( q_*(s,a) - Q_\theta(s,a) \big)^2 \Big], \qquad \mathcal{L}(\theta) = \mathbb{E}_{s,a,r,s'} \Big[ \big( r + \gamma \max_{a'} Q_\theta(s',a') - Q_\theta(s,a) \big)^2 \Big]$

**Semi-gradient** (the target is held fixed although it depends on $\displaystyle \theta$):

$\displaystyle \nabla_\theta \mathcal{L}(\theta) = -2\, \mathbb{E}_{s,a,r,s'} \Big[ \big( r + \gamma \max_{a'} Q_\theta(s',a') - Q_\theta(s,a) \big)\, \nabla_\theta Q_\theta(s,a) \Big], \qquad \theta \leftarrow \theta + \alpha\, \big( r + \gamma \max_{a'} Q_\theta(s',a') - Q_\theta(s,a) \big)\, \nabla_\theta Q_\theta(s,a)$

**DQN loss** (experience replay: transitions sampled uniformly from the memory; target network: the bootstrap uses the frozen copy). In code the target is multiplied by $\displaystyle (1-d)$, so that a terminal transition has no future value:

$\displaystyle \mathcal{L}(\theta_{\text{online}}) = \mathbb{E}_{(s,a,r,s') \sim U(\mathcal{D})} \Big[ \big( r + \gamma\, (1-d) \max_{a'} Q_{\theta_{\text{target}}}(s',a') - Q_{\theta_{\text{online}}}(s,a) \big)^2 \Big], \qquad \theta_{\text{target}} \leftarrow \theta_{\text{online}} \ \text{every } C \text{ steps}$

**Double DQN** (the online network selects the next action, the target network evaluates it; it rewrites the maximum as selection followed by evaluation):

$\displaystyle \max_{a'} Q_{\theta_{\text{target}}}(s',a') = Q_{\theta_{\text{target}}}\Big( s', \arg\max_{a'} Q_{\theta_{\text{target}}}(s',a') \Big) \quad \longrightarrow \quad y = r + \gamma\, (1-d)\, Q_{\theta_{\text{target}}}\Big( s', \arg\max_{a'} Q_{\theta_{\text{online}}}(s',a') \Big)$

**Prioritized experience replay:**

$\displaystyle p_i = \lvert \delta_i \rvert + \epsilon, \qquad P(i) = \frac{p_i^{\alpha}}{\sum_k p_k^{\alpha}}, \qquad w_i = \Big( \frac{1}{N}\, \frac{1}{P(i)} \Big)^{\beta}, \qquad w_i \leftarrow \frac{w_i}{\max_j w_j}, \qquad \mathcal{L}(\theta) = \mathbb{E}_{i \sim P(\mathcal{D})} \big[ w_i\, \delta_i^2 \big]$

**Dueling aggregation** (mean-subtracted advantages, so that the split of $\displaystyle Q$ into $\displaystyle V$ and $\displaystyle A$ is identifiable):

$\displaystyle a_\pi(s,a) = q_\pi(s,a) - v_\pi(s), \qquad Q(s,a;\theta,\theta_V,\theta_A) = V(s;\theta,\theta_V) + \Big( A(s,a;\theta,\theta_A) - \frac{1}{\lvert \mathcal{A} \rvert} \sum_{a'} A(s,a';\theta,\theta_A) \Big)$

**Deadly triad**: function approximation + bootstrapping + off-policy learning; the three are present in NFQ, DQN and in the off-policy actor–critics of Chapter 10.

**In code**: `online_q` and `target_q` hold $\displaystyle \theta_{\text{online}}$ and $\displaystyle \theta_{\text{target}}$, `failure` is $\displaystyle d$, `target` is the bootstrapped target, `td_errors` is $\displaystyle \delta$, `memory_size` is $\displaystyle N$, `batch_size` is $\displaystyle m$, `target_update_steps` is $\displaystyle C$, `alpha` and `beta` are the PER exponents; the experience tuple is `(state, action, reward, next_state, failure)`.

## Chapter 08 — Policy gradient methods

The **policy is parameterised directly** and improved by gradient ascent on a performance measure; the value function becomes optional (a baseline).

| Symbol | Meaning |
|---|---|
| $\displaystyle \pi_\theta(a \mid s)$ | **parameterised policy** with parameters $\displaystyle \theta$ (the weights of the policy network) |
| $\displaystyle h(s,a,\theta)$ | **preference** (logit) of action $\displaystyle a$ in state $\displaystyle s$; the softmax of the preferences is the policy |
| $\displaystyle x(s,a)$ | feature vector (linear preferences); replaces the $\displaystyle \phi(s,a)$ of Chapter 07 |
| $\displaystyle J(\theta)$ | **performance measure**: expected return from the start of an episode |
| $\displaystyle p_0$ | distribution of the initial state $\displaystyle S_0$ |
| $\displaystyle \tau$ | a **trajectory** $\displaystyle (S_0, A_0, R_1, S_1, \ldots, S_{T-1}, A_{T-1}, R_T, S_T)$ |
| $\displaystyle G(\tau)$ | discounted return of a whole trajectory |
| $\displaystyle p(\tau \mid \pi_\theta)$ | probability of a trajectory under the policy |
| $\displaystyle \nabla_\theta \log \pi_\theta(a \mid s)$ | **score function** |
| $\displaystyle m$ | number of sampled trajectories in a gradient estimate; $\displaystyle \tau^{(i)}, s_t^{(i)}, a_t^{(i)}$ the $\displaystyle i$-th of them |
| $\displaystyle \mu_\pi(s)$ | distribution of the states visited under $\displaystyle \pi$ (state-form of the theorem; written $\displaystyle \rho^{\mu_\theta}$ in Chapter 10 and $\displaystyle d^{\pi}$ in Chapter 11) |
| $\displaystyle G_t$ | **reward-to-go**: return from time $\displaystyle t$ |
| $\displaystyle b(S_t)$ | a **baseline**, any function of the state |
| $\displaystyle V(s;\phi)$ | **value network** with parameters $\displaystyle \phi$, the learned baseline |
| $\displaystyle A(S_t, A_t)$ | **sample advantage** $\displaystyle G_t - V(S_t;\phi)$, an estimate of $\displaystyle a_\pi(S_t,A_t)$ |
| $\displaystyle \mathcal{L}(\theta)$, $\displaystyle \mathcal{L}_\pi(\theta)$, $\displaystyle \mathcal{L}_v(\phi)$ | REINFORCE loss, policy loss and value loss (losses: descending them ascends $\displaystyle J$) |
| $\displaystyle H\big( \pi_\theta(\cdot \mid s) \big)$ | **entropy** of the action distribution in state $\displaystyle s$ |
| $\displaystyle \beta$ | **entropy coefficient** (Chapters 08–11; in Chapter 13 $\displaystyle \beta$ becomes the KL coefficient) |

**Policy, objective and trajectory probability** (the dynamics $\displaystyle p_0$ and $\displaystyle p(s' \mid s,a)$ do not depend on $\displaystyle \theta$):

$\displaystyle \pi_\theta(a \mid s) = \frac{e^{h(s,a,\theta)}}{\sum_{b} e^{h(s,b,\theta)}}, \qquad \theta \leftarrow \theta + \alpha\, \nabla_\theta J(\theta)$

$\displaystyle J(\theta) = \mathbb{E}_{S_0 \sim p_0}\big[ v_{\pi_\theta}(S_0) \big] = \mathbb{E}_{\tau \sim \pi_\theta}\big[ G(\tau) \big], \qquad G(\tau) = \sum_{t=0}^{T-1} \gamma^{t} R_{t+1}, \qquad p(\tau \mid \pi_\theta) = p_0(S_0) \prod_{t=0}^{T-1} \pi_\theta(A_t \mid S_t)\, p(S_{t+1} \mid S_t, A_t)$

**Policy gradient theorem** (log-derivative trick; the dynamics drop out of the gradient):

$\displaystyle \nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\big[ \nabla_\theta \log p(\tau \mid \pi_\theta)\, G(\tau) \big] = \mathbb{E}_{\tau \sim \pi_\theta}\left[ \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(A_t \mid S_t)\, G(\tau) \right] = \mathbb{E}_{\pi}\big[ q_\pi(S_t, A_t)\, \nabla_\theta \log \pi_\theta(A_t \mid S_t) \big]$

Score of a linear softmax policy: $\displaystyle \nabla_\theta \log \pi_\theta(a \mid s) = x(s,a) - \sum_{b} \pi_\theta(b \mid s)\, x(s,b)$. The score has zero mean: $\displaystyle \mathbb{E}_{A_t \sim \pi_\theta}\big[ \nabla_\theta \log \pi_\theta(A_t \mid S_t) \mid S_t \big] = \nabla_\theta \sum_a \pi_\theta(a \mid S_t) = 0$.

**Causality and baseline** (past rewards and any state-dependent baseline contribute zero in expectation; the factor $\displaystyle \gamma^t$ of the discounted objective is dropped, as is standard):

$\displaystyle G_t = \sum_{k=t}^{T-1} \gamma^{k-t} R_{k+1} = R_{t+1} + \gamma\, G_{t+1}, \qquad \nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[ \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(A_t \mid S_t)\, \big( G_t - b(S_t) \big) \right]$

**REINFORCE** (Monte Carlo estimate over $\displaystyle m$ trajectories; one trajectory per update in the implementation):

$\displaystyle \nabla_\theta J(\theta) \approx \frac{1}{m} \sum_{i=1}^{m} \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta\big( a_t^{(i)} \mid s_t^{(i)} \big)\, G_t\big( \tau^{(i)} \big), \qquad \mathcal{L}(\theta) = -\frac{1}{T} \sum_{t=0}^{T-1} G_t\, \log \pi_\theta(A_t \mid S_t)$

**Vanilla policy gradient (REINFORCE with baseline)**: value network regressed on the returns, policy weighted by the sample advantage (detached), entropy bonus against premature collapse:

$\displaystyle A(S_t, A_t) = G_t - V(S_t;\phi), \qquad \mathcal{L}_v(\phi) = \frac{1}{T} \sum_{t=0}^{T-1} \big( G_t - V(S_t;\phi) \big)^2, \qquad \mathcal{L}_\pi(\theta) = -\frac{1}{T} \sum_{t=0}^{T-1} \big( G_t - V(S_t;\phi) \big) \log \pi_\theta(A_t \mid S_t)$

$\displaystyle H\big( \pi_\theta(\cdot \mid s) \big) = -\sum_{a} \pi_\theta(a \mid s)\, \log \pi_\theta(a \mid s), \qquad \nabla_\theta \Big[ J(\theta) + \beta\, \mathbb{E}_{\pi_\theta}\big[ H(\pi_\theta(\cdot \mid S_t)) \big] \Big] \approx \frac{1}{T} \sum_{t=0}^{T-1} \nabla_\theta \Big[ \log \pi_\theta(A_t \mid S_t)\, A(S_t, A_t) + \beta\, H\big( \pi_\theta(\cdot \mid S_t) \big) \Big]$

The baseline is used *only to centre* the returns; the moment $\displaystyle V$ replaces part of the return (bootstrapping) the method becomes an actor–critic (Chapter 09).

**In code**: the policy network outputs logits fed to a `Categorical` distribution; `select_action` returns the action, its log-probability and the entropy; `stochastic_pi` samples, `greedy_pi` takes the most probable action; `beta` is $\displaystyle \beta$.

## Chapter 09 — Actor–critic methods

The value network becomes a **critic**: its estimate **replaces part of the return** (bootstrapping), and its evaluation, not the observed return, drives the update of the **actor**.

| Symbol | Meaning |
|---|---|
| $\displaystyle \pi_\theta$, $\displaystyle V(s;\phi)$ | **actor** (policy network) and **critic** (value network), with separate parameters and optimisers |
| $\displaystyle d_{t+1}$ | **terminal flag** of the transition $\displaystyle (S_t, A_t, R_{t+1}, S_{t+1})$: 1 only for a genuine termination, 0 for a truncation (`terminated` / `dones` in code) |
| $\displaystyle y_t$ | **critic target** (treated as a constant: semi-gradient) |
| $\displaystyle \delta_t$ | **TD error**, the one-step advantage estimate |
| $\displaystyle G_{t:t+n}$ | $\displaystyle n$-step return bootstrapped on the critic |
| $\displaystyle A^{(n)}_t$ | $\displaystyle n$-step **advantage estimate** |
| $\displaystyle n$ | number of steps of a **rollout** before bootstrapping (`n_steps`) |
| $\displaystyle K$ | number of **parallel environments** (A2C, `n_envs`) or workers (A3C) |
| $\displaystyle c_v$ | coefficient of the value loss when actor and critic share one network |
| $\displaystyle \lambda$ | GAE parameter, the trace-decay of Chapter 05 applied to advantages (`lam`) |
| $\displaystyle A^{\text{GAE}(\gamma,\lambda)}_t$ | **generalised advantage estimate** |
| $\displaystyle l$ | offset index in sums of TD errors |

**TD error as advantage estimate** (unbiased if the critic is exact):

$\displaystyle A(S_t, A_t) = R_{t+1} + \gamma\, V(S_{t+1};\phi) - V(S_t;\phi) = \delta_t, \qquad \mathbb{E}_\pi\big[ \delta_t \mid S_t = s, A_t = a \big] = q_\pi(s,a) - v_\pi(s) = a_\pi(s,a) \quad \text{if } V = v_\pi$

**One-step actor–critic** (one transition per update):

$\displaystyle y_t = R_{t+1} + \gamma\, (1 - d_{t+1})\, V(S_{t+1};\phi), \qquad \delta_t = y_t - V(S_t;\phi), \qquad \mathcal{L}_v(\phi) = \big( y_t - V(S_t;\phi) \big)^2, \qquad \text{actor: ascend } \delta_t \log \pi_\theta(A_t \mid S_t) + \beta\, H\big( \pi_\theta(\cdot \mid S_t) \big)$

**$\displaystyle n$-step returns and advantages** (A3C/A2C; the recursion is started from the bootstrap value and cut by the terminal flag):

$\displaystyle G_{t:t+n} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^{n}\, (1 - d_{t+n})\, V(S_{t+n};\phi), \qquad G_{t:t+n} = R_{t+1} + \gamma\, (1 - d_{t+1})\, G_{t+1:t+n}$

$\displaystyle A^{(n)}_t = G_{t:t+n} - V(S_t;\phi), \qquad y_t = G_{t:t+n}$

| Advantage estimate | Formula | Bias | Variance |
|---|---|---|---|
| Monte Carlo (VPG) | $\displaystyle G_t - V(S_t)$ | none | high |
| $\displaystyle n$-step | $\displaystyle G_{t:t+n} - V(S_t)$ | from the error of $\displaystyle V(S_{t+n})$, discounted by $\displaystyle \gamma^n$ | intermediate |
| one-step (TD error) | $\displaystyle R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$ | from the error of $\displaystyle V(S_{t+1})$ | low |

**A2C**: $\displaystyle K$ environments stepped in lockstep for $\displaystyle n$ steps give a mini-batch of $\displaystyle K \times n$ transitions per update (parallelism replaces the replay memory as a source of decorrelated data); gradients are clipped in norm. With a shared network:

$\displaystyle \mathcal{L}(\theta) = \mathcal{L}_\pi(\theta) + c_v\, \mathcal{L}_v(\theta) - \beta\, H$

**Generalised advantage estimation** (every $\displaystyle n$-step advantage telescopes into a sum of TD errors; GAE is their exponentially weighted average, i.e. the $\displaystyle \lambda$-return written in TD errors):

$\displaystyle \delta_{t+l} = R_{t+l+1} + \gamma\, V(S_{t+l+1}) - V(S_{t+l}), \qquad A^{(n)}_t = \sum_{l=0}^{n-1} \gamma^{l}\, \delta_{t+l}$

$\displaystyle A^{\text{GAE}(\gamma,\lambda)}_t = (1-\lambda) \sum_{n=1}^{\infty} \lambda^{n-1} A^{(n)}_t = \sum_{l=0}^{\infty} (\gamma\lambda)^{l}\, \delta_{t+l}, \qquad A^{\text{GAE}}_t = \delta_t + \gamma\lambda\, (1 - d_{t+1})\, A^{\text{GAE}}_{t+1}, \qquad y_t = A^{\text{GAE}}_t + V(S_t;\phi)$

$\displaystyle \lambda = 0 \Rightarrow \delta_t \ \text{(one-step)}, \qquad \lambda = 1 \Rightarrow \text{Monte Carlo advantage}$

**In code**: `n_envs` is $\displaystyle K$, `n_steps` is $\displaystyle n$, `gamma` and `lam` are $\displaystyle \gamma, \lambda$, `beta` is $\displaystyle \beta$, `dones` are the flags $\displaystyle d$, `last_values` the bootstrap values $\displaystyle V(S_{t+n};\phi)$, `max_gradient_norm` the clipping threshold; training is organised by environment steps (`max_steps`, `eval_every`).

## Chapter 10 — Continuous control

Actions are **vectors of real numbers**: the maximum over actions of value-based methods is unavailable, and either the stochastic policy of Chapter 08 is made Gaussian, or the critic's gradient with respect to the action tells a deterministic actor where to move (DDPG, TD3), or both ideas are combined with a maximum-entropy objective (SAC). All three are **off-policy** and reuse the replay memory and target networks of Chapter 07.

| Symbol | Meaning |
|---|---|
| $\displaystyle \mathcal{A} = [a_{\min}, a_{\max}]^{d}$ | **continuous action space**, a box of dimension $\displaystyle d$ (here $\displaystyle d$ is a dimension; the terminal flag of the same chapter is also written $\displaystyle d$, see the consistency notes) |
| $\displaystyle u \in [-2, 2]$, $\displaystyle \theta_t$, $\displaystyle \dot\theta_t$ | Pendulum: torque, angle from the upright position, angular velocity (this $\displaystyle \theta$ is an angle, not the actor's parameters) |
| $\displaystyle \mu_\theta(s)$, $\displaystyle \sigma_\theta(s)$ | **mean** and **standard deviation** of a Gaussian policy (one per action component; the network outputs $\displaystyle \log \sigma$) |
| $\displaystyle \xi \sim \mathcal{N}(0, I)$, $\displaystyle \odot$ | standard Gaussian noise and element-wise product (reparameterised sampling) |
| $\displaystyle \mu_\theta(s)$ | **deterministic actor** (DDPG, TD3): outputs the action directly; the same letter as the Gaussian mean, which SAC reuses in that sense |
| $\displaystyle Q_\phi(s,a)$ | **critic** taking state *and* action as input and returning a scalar |
| $\displaystyle \nabla_a Q_\phi(s,a)$ | gradient of the critic with respect to the action: the direction in which the actor should move its output |
| $\displaystyle \rho$, $\displaystyle \rho^{\mu_\theta}$ | a distribution over states / the (discounted) distribution of the states visited under $\displaystyle \mu_\theta$ |
| $\displaystyle \theta^-$, $\displaystyle \phi^-$ | **target copies** of the actor and critic parameters (Chapter 07 writes $\displaystyle \theta_{\text{target}}$) |
| $\displaystyle \tau$ | **soft (Polyak) update** mixing factor (`tau`, e.g. $\displaystyle 0.005$); the target tracks the online network with time constant $\displaystyle 1/\tau$ steps |
| $\displaystyle y$ | critic target; $\displaystyle (s,a,r,s',d) \sim U(\mathcal{D})$ a mini-batch from the replay memory |
| $\displaystyle \epsilon_t \sim \mathcal{N}(0, \sigma^2 I)$ | **exploration noise** added to the actor's action during data collection (`noise_std` is $\displaystyle \sigma$, a fraction of $\displaystyle a_{\max}$) |
| $\displaystyle \text{clip}(x, a_{\min}, a_{\max})$ | element-wise clipping to the action bounds |
| $\displaystyle Q_{\phi_1}, Q_{\phi_2}$ | the **twin critics** of TD3 and SAC (with targets $\displaystyle \phi_1^-, \phi_2^-$) |
| $\displaystyle \tilde a$, $\displaystyle \tilde\sigma$, $\displaystyle c$ | **smoothed target action**, standard deviation of its noise (`target_noise_std`) and clipping range of that noise (`target_noise_clip`) (TD3) |
| $\displaystyle k$ | **policy delay**: the actor and the targets are updated once every $\displaystyle k$ critic updates (TD3) |
| $\displaystyle \alpha$ | **temperature** of the maximum-entropy objective (SAC; the letter of the literature, not a step size) |
| $\displaystyle H\big( \pi(\cdot \mid s) \big)$ | entropy of the policy in state $\displaystyle s$ |
| $\displaystyle a' \sim \pi_\theta(\cdot \mid s')$ | next action **sampled from the current actor** in the SAC target (no target actor) |
| $\displaystyle f_\theta(s, \xi)$, $\displaystyle u$ | reparameterised action and pre-squashing Gaussian variable (SAC) |
| $\displaystyle \bar H$ | **target entropy** of the automatic temperature (heuristic $\displaystyle \bar H = -d$) |

**Gaussian policy for continuous actions** (what changes in the methods of Chapters 08–09: only $\displaystyle \log \pi_\theta$ and the entropy):

$\displaystyle \pi_\theta(a \mid s) = \prod_{i=1}^{d} \mathcal{N}\big( a_i;\ \mu_{\theta,i}(s),\ \sigma_{\theta,i}(s)^2 \big), \qquad a = \mu_\theta(s) + \sigma_\theta(s) \odot \xi$

$\displaystyle \log \pi_\theta(a \mid s) = \sum_{i=1}^{d} \left[ -\frac{(a_i - \mu_i)^2}{2\sigma_i^2} - \log \sigma_i - \frac{1}{2} \log 2\pi \right], \qquad H = \sum_{i=1}^{d} \Big( \log \sigma_i + \tfrac{1}{2} \log 2\pi e \Big)$

$\displaystyle \nabla_{\mu_i} \log \pi_\theta(a \mid s) = \frac{a_i - \mu_i}{\sigma_i^2}, \qquad \nabla_{\sigma_i} \log \pi_\theta(a \mid s) = \frac{(a_i - \mu_i)^2 - \sigma_i^2}{\sigma_i^3}$

**Deterministic policy gradient** (the actor learns the maximiser of the critic; the expectation is over states only, hence off-policy):

$\displaystyle \mu(s) = \arg\max_{a} Q_\phi(s,a) \ \text{(not computable)}, \qquad J(\theta) = \mathbb{E}_{s \sim \rho}\big[ Q_\phi\big( s, \mu_\theta(s) \big) \big], \qquad \nabla_\theta J(\theta) = \mathbb{E}_{s \sim \rho^{\mu_\theta}}\Big[ \nabla_\theta \mu_\theta(s)\ \nabla_a q_{\mu_\theta}(s,a) \big|_{a = \mu_\theta(s)} \Big]$

**DDPG** (DQN loss with the target actor in place of the $\displaystyle \arg\max$; actor loss; soft target updates; exploration by additive noise):

$\displaystyle y = r + \gamma\, (1-d)\, Q_{\phi^-}\big( s', \mu_{\theta^-}(s') \big), \qquad \mathcal{L}(\phi) = \mathbb{E}_{(s,a,r,s',d) \sim U(\mathcal{D})} \Big[ \big( y - Q_\phi(s,a) \big)^2 \Big], \qquad \mathcal{L}(\theta) = -\mathbb{E}_{s \sim U(\mathcal{D})}\big[ Q_\phi\big( s, \mu_\theta(s) \big) \big]$

$\displaystyle \theta^- \leftarrow \tau\, \theta + (1-\tau)\, \theta^-, \qquad \phi^- \leftarrow \tau\, \phi + (1-\tau)\, \phi^-, \qquad A_t = \text{clip}\big( \mu_\theta(S_t) + \epsilon_t,\ a_{\min},\ a_{\max} \big)$

**TD3** (clipped double Q-learning, target policy smoothing, delayed policy updates):

$\displaystyle \tilde a = \text{clip}\big( \mu_{\theta^-}(s') + \text{clip}(\epsilon, -c, c),\ a_{\min},\ a_{\max} \big), \quad \epsilon \sim \mathcal{N}(0, \tilde\sigma^2 I), \qquad y = r + \gamma\, (1-d) \min_{i=1,2} Q_{\phi_i^-}\big( s', \tilde a \big), \qquad \mathcal{L}(\theta) = -\mathbb{E}\big[ Q_{\phi_1}\big( s, \mu_\theta(s) \big) \big] \ \text{every } k \text{ critic updates}$

**SAC** (maximum-entropy objective, soft values and soft Bellman target; actor loss via the reparameterisation trick; squashed Gaussian; automatic temperature):

$\displaystyle J(\pi) = \mathbb{E}_\pi \left[ \sum_{t=0}^{\infty} \gamma^{t} \Big( R_{t+1} + \alpha\, H\big( \pi(\cdot \mid S_t) \big) \Big) \right], \qquad V(s) = \mathbb{E}_{a \sim \pi}\big[ Q(s,a) - \alpha \log \pi(a \mid s) \big], \qquad Q(s,a) = \mathbb{E}\big[ R + \gamma\, V(S') \big]$

$\displaystyle y = r + \gamma\, (1-d) \Big( \min_{i=1,2} Q_{\phi_i^-}(s', a') - \alpha \log \pi_\theta(a' \mid s') \Big), \quad a' \sim \pi_\theta(\cdot \mid s'), \qquad \mathcal{L}(\theta) = \mathbb{E}_{s \sim U(\mathcal{D})}\ \mathbb{E}_{a \sim \pi_\theta(\cdot \mid s)} \Big[ \alpha \log \pi_\theta(a \mid s) - \min_{i=1,2} Q_{\phi_i}(s,a) \Big]$

$\displaystyle a = f_\theta(s, \xi) = \tanh\big( \mu_\theta(s) + \sigma_\theta(s) \odot \xi \big)\, a_{\max}, \qquad \nabla_\theta\, \mathbb{E}_{a \sim \pi_\theta}\big[ Q(s,a) \big] = \mathbb{E}_{\xi}\Big[ \nabla_\theta f_\theta(s,\xi)\ \nabla_a Q(s,a) \big|_{a = f_\theta(s,\xi)} \Big]$

$\displaystyle \log \pi_\theta(a \mid s) = \log \mathcal{N}\big( u;\ \mu_\theta(s), \sigma_\theta(s)^2 \big) - \sum_{i=1}^{d} \log\big( a_{\max}\, (1 - \tanh^2 u_i) \big), \qquad \mathcal{L}(\alpha) = \mathbb{E}_{a \sim \pi_\theta}\big[ -\alpha \big( \log \pi_\theta(a \mid s) + \bar H \big) \big], \quad \bar H = -d$

**In code**: `tau` is $\displaystyle \tau$, `noise_std` is $\displaystyle \sigma$, `target_noise_std` and `target_noise_clip` are $\displaystyle \tilde\sigma$ and $\displaystyle c$, `start_steps` is the random warm-up of the memory, `log_alpha` holds $\displaystyle \log \alpha$ (so that $\displaystyle \alpha > 0$), `soft_update(target, online, tau)` is the Polyak update, `actor_pi` / `noisy_pi` / `sac_pi` / `sac_sample_pi` are the deterministic, noisy, mean and sampling policies.

## Chapter 11 — Proximal Policy Optimization

Stay on-policy but control **how much the policy changes** per update, measured in policy space (KL divergence) rather than in parameter space, and take **several gradient steps on the same rollout**.

| Symbol | Meaning |
|---|---|
| $\displaystyle D_{\text{KL}}\big( \pi(\cdot \mid s)\, \Vert\, \pi'(\cdot \mid s) \big)$ | **Kullback–Leibler divergence** between two action distributions in state $\displaystyle s$ |
| $\displaystyle \pi_{\theta_{\text{old}}}$, $\displaystyle \theta_{\text{old}}$ | the **old policy**, which collected the rollout, and its parameters (fixed during the epochs) |
| $\displaystyle \pi$, $\displaystyle \pi'$ | current policy and candidate new policy (in the theory, written as functions of the policy: $\displaystyle J(\pi) = J(\pi_\theta) = J(\theta)$) |
| $\displaystyle d^{\pi}$ | **discounted state distribution** under $\displaystyle \pi$ ($\displaystyle \mu_\pi$ in Chapter 08, $\displaystyle \rho^{\mu_\theta}$ in Chapter 10) |
| $\displaystyle L_\pi(\pi')$ | **surrogate objective** (importance-sampled advantage, states from $\displaystyle \pi$) |
| $\displaystyle C$ | constant of the TRPO bound (depends on $\displaystyle \gamma$ and on the largest advantage) |
| $\displaystyle \delta$ | **radius of the trust region** (TRPO; not the TD error) |
| $\displaystyle g$, $\displaystyle F$ | policy gradient and **Fisher information matrix** (Hessian of the KL) at $\displaystyle \theta_{\text{old}}$ |
| $\displaystyle r_t(\theta)$ | **probability ratio** of the action taken (Chapter 13 renames it $\displaystyle \rho_{i,t}$ to avoid the clash with rewards) |
| $\displaystyle \hat A_t$ | **advantage estimate** of the rollout (GAE), normalised within each mini-batch |
| $\displaystyle \epsilon$ | **clipping range** of the ratio (typically $\displaystyle 0.1$–$\displaystyle 0.2$; `epsilon`) |
| $\displaystyle L^{\text{CLIP}}(\theta)$ | **clipped surrogate objective** (maximised) |
| $\displaystyle \beta_{\text{KL}}$ | coefficient of the KL penalty in PPO-penalty (not used in the implementation) |
| $\displaystyle c_v$, $\displaystyle \beta$ | value-loss coefficient (shared network) and entropy coefficient |
| $\displaystyle E$ | number of **epochs** of mini-batch updates per rollout (`epochs`); `minibatch_size` the mini-batch size |
| $\displaystyle \mathbb{E}_t[\cdot]$ | empirical average over the transitions of a rollout or mini-batch |
| $\displaystyle \log \sigma$ | state-independent log-standard-deviation parameter of the Gaussian policy (continuous actions) |

**KL divergence and the step-size problem:**

$\displaystyle D_{\text{KL}}\big( \pi_{\theta_{\text{old}}}(\cdot \mid s)\, \Vert\, \pi_\theta(\cdot \mid s) \big) = \sum_a \pi_{\theta_{\text{old}}}(a \mid s) \log \frac{\pi_{\theta_{\text{old}}}(a \mid s)}{\pi_\theta(a \mid s)}$

**Performance difference lemma** (Kakade & Langford) and **surrogate objective** (importance sampling over actions, exact; state distribution of $\displaystyle \pi'$ replaced by that of $\displaystyle \pi$, approximate):

$\displaystyle J(\pi') - J(\pi) = \mathbb{E}_{\tau \sim \pi'} \left[ \sum_{t=0}^{\infty} \gamma^{t}\, a_\pi(S_t, A_t) \right] = \frac{1}{1-\gamma}\, \mathbb{E}_{s \sim d^{\pi'}}\ \mathbb{E}_{a \sim \pi'(\cdot \mid s)}\big[ a_\pi(s,a) \big]$

$\displaystyle L_\pi(\pi') = \frac{1}{1-\gamma}\, \mathbb{E}_{s \sim d^{\pi},\ a \sim \pi(\cdot \mid s)}\left[ \frac{\pi'(a \mid s)}{\pi(a \mid s)}\, a_\pi(s,a) \right], \qquad \nabla_\theta L_{\pi_{\theta_{\text{old}}}}(\pi_\theta)\Big|_{\theta = \theta_{\text{old}}} = \nabla_\theta J(\theta)\Big|_{\theta = \theta_{\text{old}}}$

**TRPO bound and trust-region problem** (natural gradient solution):

$\displaystyle J(\pi') \ge J(\pi) + L_\pi(\pi') - C \max_s D_{\text{KL}}\big( \pi(\cdot \mid s)\, \Vert\, \pi'(\cdot \mid s) \big)$

$\displaystyle \max_\theta\ L_{\pi_{\theta_{\text{old}}}}(\pi_\theta) \quad \text{s.t.} \quad \mathbb{E}_{s \sim d^{\pi_{\theta_{\text{old}}}}}\Big[ D_{\text{KL}}\big( \pi_{\theta_{\text{old}}}(\cdot \mid s)\, \Vert\, \pi_\theta(\cdot \mid s) \big) \Big] \le \delta, \qquad \theta = \theta_{\text{old}} + \sqrt{\frac{2\delta}{g^\top F^{-1} g}}\ F^{-1} g$

**PPO clipped surrogate** (a pessimistic lower bound of the surrogate: no incentive to move the ratio beyond $\displaystyle 1 \pm \epsilon$ in the direction the advantage asks for):

$\displaystyle r_t(\theta) = \frac{\pi_\theta(A_t \mid S_t)}{\pi_{\theta_{\text{old}}}(A_t \mid S_t)}, \qquad L^{\text{CLIP}}(\theta) = \mathbb{E}_t \Big[ \min\Big( r_t(\theta)\, \hat A_t,\ \text{clip}\big( r_t(\theta),\ 1-\epsilon,\ 1+\epsilon \big)\, \hat A_t \Big) \Big]$

**Full PPO loss** (critic targets $\displaystyle y_t$ are the GAE returns of Chapter 09; with separate networks the two parts have separate optimisers) and **approximate KL** diagnostic:

$\displaystyle \mathcal{L}(\theta, \phi) = -L^{\text{CLIP}}(\theta) + c_v\, \mathbb{E}_t\Big[ \big( y_t - V(S_t;\phi) \big)^2 \Big] - \beta\, \mathbb{E}_t\Big[ H\big( \pi_\theta(\cdot \mid S_t) \big) \Big], \qquad \hat D_{\text{KL}} = \mathbb{E}_t\big[ (r_t - 1) - \log r_t \big]$

**In code**: `epochs` is $\displaystyle E$, `epsilon` is $\displaystyle \epsilon$, `beta` is $\displaystyle \beta$, `lam` is the GAE $\displaystyle \lambda$, the rollout stores the old log-probabilities (denominators of $\displaystyle r_t$), `n_envs` and `n_steps` are $\displaystyle K$ and $\displaystyle n$ as in Chapter 09; for continuous actions the policy module exposes `log_std`.

## Chapter 12 — Offline RL and Decision Transformers

Learning from a **fixed dataset** with no further interaction. Bootstrapped methods suffer **extrapolation error** on actions the data do not support; CQL makes the values conservative; the Decision Transformer replaces value learning by return-conditioned sequence modelling.

| Symbol | Meaning |
|---|---|
| $\displaystyle \mathcal{D} = \{ (s_i, a_i, r_i, s'_i, d_i) \}_{i=1}^{N}$ | the **dataset** of $\displaystyle N$ transitions (with episode boundaries recorded separately) |
| $\displaystyle \pi_\beta$ | **behaviour policy** that generated the dataset (unknown; $\displaystyle \pi_{\text{behaviour}}$ in Chapter 05) |
| $\displaystyle \epsilon$ | probability with which the heuristic controller's action is replaced by a random one when building the datasets (expert $\displaystyle \epsilon = 0$, medium $\displaystyle 0.2$, random $\displaystyle 1$) |
| $\displaystyle \eta$, $\displaystyle T$ | per-step error probability of a cloned policy and horizon (compounding-error bound $\displaystyle \eta T^2$) |
| $\displaystyle q_{\pi_\beta}(s,a)$ | value of the behaviour policy, estimated by the discounted returns observed in the data |
| $\displaystyle y$ | bootstrapped target of offline (Double) DQN |
| $\displaystyle \alpha$ | coefficient of the **conservative penalty** (CQL; not a learning rate) |
| $\displaystyle \mathcal{L}_{\text{CQL}}(\theta)$ | CQL loss |
| $\displaystyle L$, $\displaystyle x_i \in \mathbb{R}^{E}$, $\displaystyle E$ | number of tokens of a sequence, embedding of token $\displaystyle i$, embedding size (`embedding_size`) |
| $\displaystyle W_Q, W_K, W_V$; $\displaystyle q_i, k_i, v_i$ | query, key and value projections and the resulting vectors of token $\displaystyle i$ (attention; not action values) |
| $\displaystyle w_{ij}$, $\displaystyle z_i$ | attention weight of token $\displaystyle i$ on token $\displaystyle j$ and output of the attention for token $\displaystyle i$ |
| $\displaystyle \mathbf{Q}, \mathbf{K}, \mathbf{V}$ | bold matrices of stacked queries, keys and values (bold to distinguish them from action values) |
| $\displaystyle d$ | dimension of the keys (attention); also the terminal flag $\displaystyle d_i$ of the dataset |
| $\displaystyle \hat R_t$ | **return-to-go**: undiscounted sum of the rewards from $\displaystyle t$ to the end of the episode (the hat is the Decision Transformer paper's notation, not an estimate) |
| $\displaystyle K$ | **context length**: number of time steps fed to the Decision Transformer (`context`) |
| $\displaystyle \tau$ | a trajectory written as the token sequence $\displaystyle (\hat R_0, S_0, A_0, \hat R_1, S_1, A_1, \ldots)$ |

**Behaviour cloning** (supervised maximum likelihood on the state–action pairs; ignores the rewards):

$\displaystyle \mathcal{L}(\theta) = -\frac{1}{N} \sum_{i=1}^{N} \log \pi_\theta(a_i \mid s_i)$

**Extrapolation error** (the maximum in the target ranges over actions the data never contain; online, exploration corrects them, offline nothing does):

$\displaystyle y = r + \gamma \max_{a'} Q(s', a')$

**Conservative Q-learning** (push down the soft maximum over all actions, push up the actions in the data; the gradient of the log-sum-exp is the softmax over actions):

$\displaystyle \mathcal{L}_{\text{CQL}}(\theta) = \alpha\, \mathbb{E}_{s \sim \mathcal{D}}\Big[ \log \sum_{a} e^{Q_\theta(s,a)} - \mathbb{E}_{a \sim \pi_\beta(\cdot \mid s)}\big[ Q_\theta(s,a) \big] \Big] + \frac{1}{2}\, \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}}\Big[ \big( y - Q_\theta(s,a) \big)^2 \Big]$

**Self-attention** (scaled dot-product; causal mask for prediction; several heads in parallel):

$\displaystyle q_i = W_Q x_i, \quad k_i = W_K x_i, \quad v_i = W_V x_i, \qquad w_{ij} = \frac{ \exp\big( q_i^\top k_j / \sqrt{d} \big) }{ \sum_{j'} \exp\big( q_i^\top k_{j'} / \sqrt{d} \big) }, \qquad z_i = \sum_{j} w_{ij}\, v_j, \qquad \mathbf{Z} = \text{softmax}\big( \mathbf{Q} \mathbf{K}^\top / \sqrt{d} \big) \mathbf{V}$

**Decision Transformer** (return-to-go before each state; predict the action from the state token; prompt with a desired return and decrement it by the rewards received):

$\displaystyle \hat R_t = \sum_{t' = t}^{T-1} R_{t'+1}, \qquad \tau = \big( \hat R_0, S_0, A_0,\ \hat R_1, S_1, A_1,\ \ldots,\ \hat R_{T-1}, S_{T-1}, A_{T-1} \big), \qquad P\big( A_t \mid \hat R_t, S_t, \text{previous } K-1 \text{ steps} \big), \qquad \hat R_{t+1} = \hat R_t - R_{t+1}$

**In code**: `context` is $\displaystyle K$, `embedding_size` is $\displaystyle E$, `return_scale` divides the returns-to-go, `target_return` is the prompted $\displaystyle \hat R_0$; the offline DQN uses the Chapter 07 machinery on a memory that is never refilled.

## Chapter 13 — Reinforcement learning for language models

A language model is a **policy over tokens**; a response is an episode; the reward is a scalar judgement of the whole response, learned from comparisons (reward model) or computed by a verifier; the policy is kept close to its starting point by a **KL penalty**.

| Symbol | Meaning |
|---|---|
| $\displaystyle x$ | a **prompt** |
| $\displaystyle y = (y_1, \ldots, y_T)$, $\displaystyle y_{<t}$ | a **response** of $\displaystyle T$ tokens and the tokens generated before step $\displaystyle t$ |
| $\displaystyle \mathcal{V}$ | **vocabulary**; the action space ($\displaystyle \lvert \mathcal{V} \rvert^T$ possible responses) |
| $\displaystyle \pi_\theta(y_t \mid x, y_{<t})$ | the model's **next-token distribution**: the policy, with state $\displaystyle (x, y_{<t})$ and action $\displaystyle y_t$ |
| $\displaystyle \pi_\theta(y \mid x)$ | probability of a whole response (product of the token probabilities) |
| $\displaystyle \pi_{\text{ref}}$ | **reference model** (the supervised fine-tuned model from which RL starts; frozen) |
| $\displaystyle r(x,y)$, $\displaystyle r_\phi(x,y)$ | reward of a response / **reward model** with parameters $\displaystyle \phi$ (there is no critic in this chapter) |
| $\displaystyle (x, y_w, y_l)$ | a **comparison**: prompt, preferred (winning) and rejected (losing) response; $\displaystyle y_w \succ y_l$ |
| $\displaystyle \sigma(\cdot)$ | the **logistic sigmoid** $\displaystyle 1/(1+e^{-z})$ (not a standard deviation) |
| $\displaystyle \beta$ | **KL coefficient** (price of divergence from the reference; in Chapters 08–11 $\displaystyle \beta$ was the entropy coefficient) |
| $\displaystyle D_{\text{KL}}\big( \pi_\theta(\cdot \mid x)\, \Vert\, \pi_{\text{ref}}(\cdot \mid x) \big)$ | divergence between the policy and the reference over responses (a sum of per-token divergences) |
| $\displaystyle \pi^*$, $\displaystyle Z(x)$ | optimal policy of the KL-regularised objective and its partition function |
| $\displaystyle r_\theta(x,y)$, $\displaystyle \hat r_\theta(x,y)$ | **implicit reward** of a policy (DPO) and the same without the $\displaystyle \beta \log Z(x)$ term |
| $\displaystyle G$ | **group size**: number of responses sampled per prompt (GRPO, `group_size`) |
| $\displaystyle r_i$, $\displaystyle \hat A_i$, $\displaystyle T_i$ | reward, group-standardised advantage and length of the $\displaystyle i$-th response of a group |
| $\displaystyle \rho_{i,t}(\theta)$ | per-token probability ratio (the $\displaystyle r_t(\theta)$ of Chapter 11, renamed) |
| $\displaystyle \hat D_{\text{KL}, t}$ | per-token KL estimator (the $\displaystyle k_3$ estimator) |

**Response probability** (the quantity every algorithm of the chapter manipulates):

$\displaystyle \pi_\theta(y \mid x) = \prod_{t=1}^{T} \pi_\theta(y_t \mid x, y_{<t}), \qquad \log \pi_\theta(y \mid x) = \sum_{t=1}^{T} \log \pi_\theta(y_t \mid x, y_{<t})$

**Reward model** (Bradley–Terry model of the comparisons; logistic regression on the score difference):

$\displaystyle P(y_w \succ y_l \mid x) = \sigma\big( r_\phi(x, y_w) - r_\phi(x, y_l) \big), \qquad \mathcal{L}(\phi) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}}\Big[ \log \sigma\big( r_\phi(x, y_w) - r_\phi(x, y_l) \big) \Big]$

**RLHF objective** (expected reward minus KL penalty; the penalty is charged token by token as a negative reward, with the course timing convention $\displaystyle R_{t+1}$ for the action $\displaystyle A_t = y_t$):

$\displaystyle \max_\theta\ \mathbb{E}_{x \sim \mathcal{D},\ y \sim \pi_\theta(\cdot \mid x)}\Big[ r_\phi(x, y) - \beta\, \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)} \Big] = \mathbb{E}_x\Big[ \mathbb{E}_{y \sim \pi_\theta}[r_\phi(x,y)] - \beta\, D_{\text{KL}}\big( \pi_\theta(\cdot \mid x)\, \Vert\, \pi_{\text{ref}}(\cdot \mid x) \big) \Big]$

$\displaystyle R_{t+1} = -\beta \Big( \log \pi_\theta(y_t \mid x, y_{<t}) - \log \pi_{\text{ref}}(y_t \mid x, y_{<t}) \Big) \ \ (t < T), \qquad R_{T+1} = r_\phi(x, y) - \beta \Big( \log \pi_\theta(y_T \mid x, y_{<T}) - \log \pi_{\text{ref}}(y_T \mid x, y_{<T}) \Big)$

PPO is then applied with per-token ratios, a response-level advantage (the total reward of the response, centred and scaled across the batch, assigned to every token) and an entropy bonus. **Reward hacking**: optimising the learned reward where it is wrong.

**DPO** (closed-form optimal policy of the KL-regularised objective, inverted to express the reward through the policy; the partition function cancels in comparisons):

$\displaystyle \pi^*(y \mid x) = \frac{1}{Z(x)}\, \pi_{\text{ref}}(y \mid x)\, \exp\Big( \frac{r(x,y)}{\beta} \Big), \qquad r_\theta(x,y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x)$

$\displaystyle \mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}}\Big[ \log \sigma\Big( \beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \Big) \Big]$

$\displaystyle \nabla_\theta \mathcal{L}_{\text{DPO}} = -\beta\, \mathbb{E}\Big[ \sigma\big( \hat r_\theta(x, y_l) - \hat r_\theta(x, y_w) \big) \Big( \nabla_\theta \log \pi_\theta(y_w \mid x) - \nabla_\theta \log \pi_\theta(y_l \mid x) \Big) \Big]$

**GRPO** (PPO without a critic: the group mean is the baseline; KL penalty added to the loss with a per-token estimator):

$\displaystyle \hat A_i = \frac{ r_i - \text{mean}(r_1, \ldots, r_G) }{ \text{std}(r_1, \ldots, r_G) }, \qquad \hat D_{\text{KL}, t} = \frac{\pi_{\text{ref}}(y_t \mid x, y_{<t})}{\pi_\theta(y_t \mid x, y_{<t})} - \log \frac{\pi_{\text{ref}}(y_t \mid x, y_{<t})}{\pi_\theta(y_t \mid x, y_{<t})} - 1$

$\displaystyle \mathcal{L}_{\text{GRPO}}(\theta) = -\frac{1}{G} \sum_{i=1}^{G} \frac{1}{T_i} \sum_{t=1}^{T_i} \Big[ \min\Big( \rho_{i,t}(\theta)\, \hat A_i,\ \text{clip}\big( \rho_{i,t}(\theta), 1-\epsilon, 1+\epsilon \big)\, \hat A_i \Big) - \beta\, \hat D_{\text{KL}, i, t} \Big]$

**In code**: `reference` is $\displaystyle \pi_{\text{ref}}$, `reward_model` is $\displaystyle r_\phi$, `beta` is $\displaystyle \beta$, `group_size` is $\displaystyle G$, `epsilon` the PPO clipping range, `response_mask` marks the tokens that count as actions, `sequence_log_prob` computes $\displaystyle \log \pi_\theta(y \mid x)$, `response_log_ratio` the log-ratio to the reference, `verify` is the verifier.

## Networks and their parameters across chapters

| Chapter | Policy / actor | Value / critic | Target copies | Other learned objects |
|---|---|---|---|---|
| 07 | implicit (greedy in $\displaystyle Q$) | $\displaystyle Q_\theta(s,a)$, $\displaystyle Q(s,a;\theta)$; dueling heads $\displaystyle V(s;\theta,\theta_V)$, $\displaystyle A(s,a;\theta,\theta_A)$ | $\displaystyle \theta_{\text{target}}$ (vs. $\displaystyle \theta_{\text{online}}$), hard update every $\displaystyle C$ steps | — |
| 08 | $\displaystyle \pi_\theta(a \mid s)$ (softmax of $\displaystyle h(s,a,\theta)$) | baseline $\displaystyle V(s;\phi)$ | — | — |
| 09 | $\displaystyle \pi_\theta(a \mid s)$ (actor) | critic $\displaystyle V(s;\phi)$ | — | — |
| 10 | $\displaystyle \mu_\theta(s)$ (deterministic) or squashed Gaussian $\displaystyle \pi_\theta$ with $\displaystyle \mu_\theta(s), \sigma_\theta(s)$ | $\displaystyle Q_\phi(s,a)$; twins $\displaystyle Q_{\phi_1}, Q_{\phi_2}$ | $\displaystyle \theta^-$, $\displaystyle \phi^-$ ($\displaystyle \phi_i^-$), soft update with $\displaystyle \tau$ | temperature $\displaystyle \alpha$ (as $\displaystyle \log \alpha$) |
| 11 | $\displaystyle \pi_\theta$, with $\displaystyle \pi_{\theta_{\text{old}}}$ the rollout policy | critic $\displaystyle V(s;\phi)$ | — | $\displaystyle \log \sigma$ (continuous actions) |
| 12 | $\displaystyle \pi_\theta$ (cloned); $\displaystyle Q_\theta$ (offline DQN, CQL); Decision Transformer | — | target network of DQN | $\displaystyle \pi_\beta$ (unknown behaviour policy) |
| 13 | $\displaystyle \pi_\theta(y_t \mid x, y_{<t})$ (language model) | — (no critic; group baseline in GRPO) | $\displaystyle \pi_{\text{ref}}$ (frozen reference) | reward model $\displaystyle r_\phi(x,y)$ |

## Overloaded symbols

Letters whose meaning depends on the chapter. Context always disambiguates, but the table makes the overloads explicit; the most frequent are $\displaystyle \alpha$, $\displaystyle \beta$, $\displaystyle \tau$, $\displaystyle \theta$, $\displaystyle d$, $\displaystyle \epsilon$ and $\displaystyle \mu$.

| Symbol | Meanings (chapter) |
|---|---|
| $\displaystyle t$ | continuous time (01); discrete time step (02 on); token position in a response (13) |
| $\displaystyle T$ | maximum percept-sequence length (01); final step of an episode (02 on); bandit horizon (04); response length (13) |
| $\displaystyle P$ | number of percepts (01); probability of an event (02 on); priority of a pair (06); sampling probability $\displaystyle P(i)$ (07) |
| $\displaystyle p$ | dynamics function (02 on); posterior density (04); exponent in $\displaystyle \alpha_n = 1/n^p$ (04); trajectory probability $\displaystyle p(\tau \mid \pi_\theta)$ and initial distribution $\displaystyle p_0$ (08); priority $\displaystyle p_i$ (07) |
| $\displaystyle k$ | summation index in $\displaystyle G_t$ (02); sweep counter (03); number of arms (04); update counter of a pair (05); policy delay (10); key vector $\displaystyle k_i$ (12) |
| $\displaystyle K$ | number of improvement steps (03); number of parallel environments or workers (09, 11); context length (12) |
| $\displaystyle n$ | number of selections of an action (04); number of planning steps (06); number of features (07); number of real rewards before bootstrapping (05, 09, 11) |
| $\displaystyle N$ | $\displaystyle N_{\text{BF}}, N_{\text{PI}}$ (03); selection counts $\displaystyle N_t(a)$, $\displaystyle N(s)$, $\displaystyle N(s,a,s')$ (04–06); size of the replay memory (07); size of the dataset (12) |
| $\displaystyle m$ | mass of a boid (01); number of sampled trajectories (08); mini-batch size (07) |
| $\displaystyle \theta$ | stopping tolerance (03); priority threshold (06); network parameters (07 on); pendulum angle (10) |
| $\displaystyle \phi$ | feature vector $\displaystyle \phi(s,a)$ (07); critic / value-network parameters (08–11); reward-model parameters (13) |
| $\displaystyle \alpha$ | step size (04–06, 08); prioritisation exponent (07); SAC temperature (10); CQL coefficient (12) |
| $\displaystyle \beta$ | step size of the recency-weighted reward model (06); importance-sampling exponent (07); entropy coefficient (08–11); KL coefficient (13); behaviour policy $\displaystyle \pi_\beta$ (12) |
| $\displaystyle \gamma$ | discount factor everywhere |
| $\displaystyle \lambda$ | Lagrange multiplier (06); precision (04); trace-decay / $\displaystyle \lambda$-return / GAE parameter (05, 09, 11) |
| $\displaystyle \tau$ | temperature (04); time since last visit $\displaystyle \tau(s,a)$ (06); trajectory (08–11, 12); Polyak factor (10) |
| $\displaystyle \mu$ | posterior mean $\displaystyle \mu_t(a)$ (04); state distribution $\displaystyle \mu_\pi(s)$ (08); deterministic actor $\displaystyle \mu_\theta(s)$ and Gaussian mean (10) |
| $\displaystyle \sigma$ | reward standard deviation (04); exploration-noise and policy standard deviation (10); logistic sigmoid (13) |
| $\displaystyle \rho$ | state distribution (10); per-token probability ratio $\displaystyle \rho_{i,t}$ (13) |
| $\displaystyle \varepsilon$ vs. $\displaystyle \epsilon$ | $\displaystyle \varepsilon$: exploration probability (04–07); $\displaystyle \epsilon$: priority offset (07), exploration noise $\displaystyle \epsilon_t$ (10), PPO clipping range (11, 13), action-corruption probability (12) |
| $\displaystyle \delta$ | $\displaystyle \delta_t$ TD error (05 on); $\displaystyle \delta(s,a)$ Bellman residual (06); trust-region radius (11) |
| $\displaystyle \Delta$ | $\displaystyle \Delta t$ integration step (01); $\displaystyle \Delta_k$ largest change in a sweep (03); $\displaystyle \Delta_a$ gap of an action (04) |
| $\displaystyle d$ | terminal flag (07, 09–12); action dimension (10); key dimension (12); discounted state distribution $\displaystyle d^{\pi}$ (11) |
| $\displaystyle c$, $\displaystyle C$ | UCB coefficient (04); target-noise clip (10); $\displaystyle c_v$ value-loss coefficient (09, 11); $\displaystyle C$ cost (03), target-update period (07), TRPO constant (11) |
| $\displaystyle E$ | lookup-table entries (01); eligibility trace $\displaystyle E(s)$, $\displaystyle E(s,a)$ (05); number of epochs (11); embedding size (12) |
| $\displaystyle L$ | number of evaluation sweeps (03); $\displaystyle L_T$ cumulative regret (04); surrogate objectives $\displaystyle L_\pi(\pi')$, $\displaystyle L^{\text{CLIP}}$ (11); number of tokens (12) |
| $\displaystyle \mathcal{L}$ | loss to be minimised (07 on); log-likelihood $\displaystyle \log \mathcal{L}(\theta)$ (06) |
| $\displaystyle \mathcal{D}$ | evidence $\displaystyle \mathcal{D}_t$ (04); replay memory (07, 10); fixed dataset (12); comparison dataset / prompt distribution (13) |
| $\displaystyle r$ | reward value (02 on); expected reward $\displaystyle r(s,a)$, $\displaystyle r(s,a,s')$ (02); probability ratio $\displaystyle r_t(\theta)$ (11); reward of a response $\displaystyle r(x,y)$, $\displaystyle r_\phi$, $\displaystyle r_i$ (13) |
| $\displaystyle q, v$ | true value functions (03 on); query and value vectors $\displaystyle q_i, v_i$ of attention (12) |
| $\displaystyle Q_0$, $\displaystyle Q_1$, $\displaystyle Q_2$ | $\displaystyle Q_0$ optimistic initial value / prior mean (04); $\displaystyle Q_1$ initial estimate (04) or first table of Double Q-learning (05); $\displaystyle Q_2$ second table (05) |
| $\displaystyle A$ | action random variable $\displaystyle A_t$; advantage estimate $\displaystyle A(S_t,A_t)$, $\displaystyle A^{(n)}_t$, $\displaystyle \hat A_t$ (07–13); advantage head (07); $\displaystyle A^*$ selected action (05) |
| $\displaystyle H$ | entropy (08–11, 10 soft values); target entropy $\displaystyle \bar H$ (10) |
| $\displaystyle x$ | feature vector $\displaystyle x(s,a)$ (08); token embedding $\displaystyle x_i$ (12); prompt (13) |
| $\displaystyle y$ | bootstrapped target (07, 09–12); response (13) |
| $\displaystyle u$ | UCB bonus $\displaystyle u_t(a)$ (04); torque / pre-squashing Gaussian variable (10) |

## Consistency notes

Deviations from the conventions above that are present in the current chapter notes, chapter by chapter. The convention recommended here is stated in each case.

**Chapters 01–04**

1. **Ch. 03, policy iteration**: the gap is written $\displaystyle \text{gap}_k = \max_s \lvert V_*(s) - V_k(s) \rvert$ with an uppercase $\displaystyle V_*$; by rule 3 it should be $\displaystyle v_*$.
2. **Ch. 04, Softmax**: probabilities are written $\displaystyle \Pr(A_t = a)$, while the rest of the course uses $\displaystyle P(A_t = a)$.
3. **Ch. 04, initial estimate**: $\displaystyle Q_1$ in the incremental derivations, $\displaystyle Q_0$ in optimistic initialisation and Thompson sampling; in Ch. 05 $\displaystyle Q_1, Q_2$ are the two tables of Double Q-learning. Suggested: $\displaystyle Q_1$ for the initial estimate everywhere in Ch. 04 (it is the estimate *before* the first reward) and $\displaystyle Q_A, Q_B$ for the two tables of Ch. 05.
4. **argmax**: $\displaystyle \underset{a}{\operatorname{argmax}}$, $\displaystyle \underset{a}{\arg\max}$ and $\displaystyle \arg\max_a$ coexist (Ch. 03, 05); $\displaystyle \arg\max_a$ is used here. Prefer $\displaystyle \in$ when ties are possible.
5. **Ch. 02, "Comparing policies"**: the policy definition is typeset as `$\pi(a|s) = P(A_t=a | S_t=s)$`, without `\displaystyle` and with `|` instead of `\mid`.
6. **Ch. 04, Thompson sampling**: $\displaystyle p\big( q(a) \mid \mathcal{D}_t \big)$ reuses the letter $\displaystyle p$ of the dynamics for a posterior density; a one-line remark would avoid confusion.

**Chapters 05–07**

7. **Ch. 05, returns**: the complete return is written $\displaystyle G_{t:T}$ throughout, while Ch. 02–03 and Ch. 08 on write $\displaystyle G_t$. The explicit horizon is useful in Ch. 05 (it shows where each target stops); state once that $\displaystyle G_{t:T} = G_t$.
8. **Ch. 05, typesetting**: several formulas contain a stray comma where a thin space was intended, e.g. `\underset{a}{\max},Q(S_{t+1},a)`, `\arg\max,Q`, `Q\leftarrow Q+\alpha,\delta_t E` (should be `\,`), and the GLIE condition is typeset `\arg\max;Q_t(s,a)`.
9. **Ch. 05, Double Q-learning**: the selected action is written $\displaystyle A^*$ (uppercase, star superscript); by rule 3 a star denotes an optimal object, and the selected action is not optimal. Suggested: $\displaystyle A' $ or $\displaystyle a_{\text{sel}}$.
10. **Ch. 05, behaviour policy**: $\displaystyle \pi_{\text{behaviour}}$ (Ch. 05) vs. $\displaystyle \pi_\beta$ (Ch. 12). Either is fine; a cross-reference between the two chapters is enough.
11. **Ch. 05, step sizes**: $\displaystyle \alpha$ (MC, traces), $\displaystyle \alpha_t$ (SARSA, Q-learning updates) and $\displaystyle \alpha_k(s,a)$ (convergence conditions) denote the same schedule; the index $\displaystyle t$ vs. $\displaystyle k$ could be unified ($\displaystyle k$ counts the updates of a pair, which is what the Robbins–Monro conditions refer to).
12. **Ch. 06, likelihood**: $\displaystyle \mathcal{L}(\theta)$ denotes a likelihood (to be maximised) in the maximum-likelihood derivation, while from Ch. 07 on $\displaystyle \mathcal{L}$ is a loss (to be minimised) and $\displaystyle \theta$ network parameters. Local and clearly introduced, but worth a remark.
13. **Ch. 07, two spellings**: $\displaystyle Q_\theta(s,a)$ and $\displaystyle Q(s,a;\theta)$ are both used in the same section (linear approximation). Suggested: one spelling per chapter (the subscript, $\displaystyle Q_\theta$, matches $\displaystyle \pi_\theta$ and $\displaystyle Q_\phi$ of later chapters; the semicolon matches $\displaystyle V(s;\phi)$ of Ch. 08–11).
14. **Ch. 07 vs. Ch. 10, target networks**: Ch. 07 writes $\displaystyle \theta_{\text{online}}, \theta_{\text{target}}$ and Ch. 10 writes $\displaystyle \theta^-, \phi^-$ for the same concept. Suggested: $\displaystyle \theta^-$ everywhere (Sutton & Barto, DQN paper), with "target network" spelled out in words in Ch. 07.
15. **Ch. 07, terminal flag**: the loss formulas omit the factor $\displaystyle (1-d)$ that the code implements (`failure`) and that Ch. 09–10 write explicitly. Adding it in Ch. 07 would make the convention visible where it is introduced.
16. **Ch. 07, $\displaystyle n$**: number of features in the linear section, while Ch. 05 and Ch. 09 use $\displaystyle n$ for the number of steps before bootstrapping; harmless but worth a word.

**Chapters 08–11**

17. **Advantage estimates**: three notations coexist, $\displaystyle A(S_t, A_t)$ (Ch. 08–09), $\displaystyle A^{(n)}_t$ (Ch. 09) and $\displaystyle \hat A_t$ (Ch. 11, 13). Suggested: $\displaystyle \hat A_t$ for every *estimate* (with $\displaystyle \hat A^{(n)}_t$, $\displaystyle \hat A^{\text{GAE}}_t$ when the estimator matters), reserving $\displaystyle a_\pi$ for the true advantage and the uppercase $\displaystyle A$ without a hat for the advantage head of Ch. 07.
18. **State distribution**: $\displaystyle \mu_\pi(s)$ (Ch. 08), $\displaystyle \rho^{\mu_\theta}$ (Ch. 10) and $\displaystyle d^{\pi}$ (Ch. 11) denote the same object. Suggested: $\displaystyle d^{\pi}$ (TRPO/PPO literature) in all three chapters, which also frees $\displaystyle \mu$ for the deterministic actor and $\displaystyle \rho$ for the probability ratio.
19. **Ch. 10, $\displaystyle \theta$ and $\displaystyle d$**: the Pendulum reward uses $\displaystyle \theta_t$ for the angle in a chapter where $\displaystyle \theta$ is the actor's parameters, and $\displaystyle d$ is both the action dimension ($\displaystyle [a_{\min}, a_{\max}]^d$, $\displaystyle \bar H = -d$) and the terminal flag ($\displaystyle 1-d$). Suggested: $\displaystyle \vartheta_t$ for the angle and $\displaystyle d_{\mathcal{A}}$ (or $\displaystyle n_a$) for the action dimension.
20. **Ch. 10, $\displaystyle \mu_\theta(s)$**: deterministic action (DDPG, TD3) and Gaussian mean (SAC); the chapter notes the double use, which is standard in the literature.
21. **Ch. 10, noise symbols**: $\displaystyle \epsilon_t$ (exploration noise), $\displaystyle \epsilon$ (target noise) and $\displaystyle \xi$ (reparameterisation noise) are three Gaussian noises with three symbols; $\displaystyle \xi$ for all of them would avoid the clash of $\displaystyle \epsilon$ with the PPO clipping range of the next chapter.
22. **Ch. 11, $\displaystyle r_t(\theta)$**: the probability ratio shares its letter with the reward; Ch. 13 renames it $\displaystyle \rho_{i,t}(\theta)$ for this reason. Suggested: $\displaystyle \rho_t(\theta)$ already in Ch. 11 (it also matches the GRPO objective).
23. **Ch. 11, $\displaystyle \delta$**: the trust-region radius reuses the letter of the TD error; $\displaystyle \delta_{\text{KL}}$ or $\displaystyle \Delta_{\text{KL}}$ would avoid it.
24. **Ch. 11, $\displaystyle E$**: number of epochs, while $\displaystyle E$ is the eligibility trace in Ch. 05; harmless.

**Chapters 12–13**

25. **Ch. 12, $\displaystyle \hat R_t$**: the return-to-go carries a hat although it is an observed quantity, not an estimate (rule 3); it equals the undiscounted $\displaystyle G_t$. A remark "$\displaystyle \hat R_t$ is the notation of the Decision Transformer paper; it is $\displaystyle G_t$ with $\displaystyle \gamma = 1$" would reconcile the two.
26. **Ch. 12, attention**: $\displaystyle q_i, k_i, v_i$ clash with the value functions $\displaystyle q_\pi, v_\pi$ and with the number of arms $\displaystyle k$; the chapter uses bold $\displaystyle \mathbf{Q}, \mathbf{K}, \mathbf{V}$ for the matrices, and the same could be done for the vectors ($\displaystyle \mathbf{q}_i, \mathbf{k}_i, \mathbf{v}_i$).
27. **Ch. 12, $\displaystyle d$**: key dimension in $\displaystyle \sqrt{d}$ and terminal flag $\displaystyle d_i$ in the dataset definition of the same chapter; $\displaystyle d_k$ (the transformer paper's notation) for the key dimension would separate them.
28. **Ch. 13, $\displaystyle \pi^*$**: the optimal policy of the KL-regularised objective is written with a star *superscript*, while Ch. 03 writes optimal objects with a star *subscript* ($\displaystyle \pi_*, v_*, q_*$). Suggested: $\displaystyle \pi_*$, or a remark that $\displaystyle \pi^*$ is the optimum of a different (regularised) objective.
29. **Ch. 13, $\displaystyle \beta$ and $\displaystyle \phi$**: both are reused with new meanings (KL coefficient, reward-model parameters); the chapter says so explicitly, which is enough.
30. **Ch. 13, $\displaystyle \sigma$**: the logistic sigmoid shares its letter with the standard deviations of Ch. 04 and 10; local and standard.